<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>Gaussian Processes</h1>



<h2>Structure</h2>

| Part | Topic | Type |
|:----:|-------|------|
| 0 | Recap | Conceptual + Math |
| 1 | tinygp | Worked example |
| 2 | Hyperparameter Learning | Math + Code |
| 3 | Exercises | Hands-on |
| 3.1 | Recover the Rotation Period | Connected |
| 3.2 | What Happens with the Wrong Kernel? | Connected |
| 3.3 | Model Comparison via the Log Marginal Likelihood | Connected |
| 3.4 | Transit Fitting with Correlated Noise | Standalone |
| 3.5 | Spectral Continuum Normalisation | Standalone |
| 3.6 | Gaussian Processes in Two Dimensions: IFU Velocity Fields | Standalone |


<h2>Library Imports</h2>
Please make sure that you have the following packages imported before the tutorial. It may be a wise idea to start a new python environment for this tutorial. You can install packages within jupyter cells by running <code>!pip install tinygp</code> for example.
</div>

In [ ]:
# Standard Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

# Cool and Important Maths Library
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True) # Turn on double precision

# Our Gaussian Process Library
import tinygp
from tinygp import kernels
from tinygp import GaussianProcess

# Scipy
from scipy.optimize import minimize as scipy_minimize
from scipy.ndimage import median_filter, binary_dilation


# NumPyro imports
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
import arviz as az

# Other
import corner
import time

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
 
<h1>§0. Recap</h1>
 
This is a recap of the `01_BeforeTutorial.ipynb` notebook. It aims to summarise the key points of that notebook, but is not a replacement for actually reading it. I strongly encourage you to read it, as it goes into much more depth than this recap.

<h2> 0.1 Generalised $\chi^2$ </h2>

The log-likelihood of observing a number of data points, $N$, that are independant with Gaussian noise is:

$$\ln \mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{2} \sum_{i=1}^{N} \left[ \frac{(y_i - m_i(\boldsymbol{\theta}))^2}{\sigma_i^2} + \ln(2\pi\sigma_i^2) \right]$$

where $y_i$ is the data, and $m_i(\boldsymbol{\theta})$ is the model prediction. From this, one can derive $\chi^2$:

$$\chi^2(\boldsymbol{\theta}) = \sum_{i=1}^{N} \frac{(y_i - m_i(\boldsymbol{\theta}))^2}{\sigma_i^2}$$

which is a goodness-of-fit metric. Note that this can be written in matrix form as:

$$ \chi^2 = \frac{1}{\mathbf{\sigma}^2}\, \mathbf{r}^T \mathbf{r} $$

where $\mathbf{r}$ is the residual vector: $\mathbf{r} = \mathbf{y} - \mathbf{m}(\boldsymbol{\theta})$

The generalised $\chi^2$ equation is:

$$ \boxed{\chi^2 = \mathbf{r}^T \mathbf{C}^{-1} \mathbf{r}} $$

where $\mathbf{C}$ is the **covariance matrix**. The log-likelihood of the generalised $\chi^2$ is then: 

$$\boxed{\ln \mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{2} \left[ \underbrace{\mathbf{r}^T \mathbf{C}^{-1} \mathbf{r}}_{\text{data-fit (generalised } \chi^2\text{)}} + \underbrace{\ln \det \mathbf{C}}_{\text{normalisation}} + \underbrace{N \ln 2\pi}_{\text{constant}} \right]}$$

Note that the $\ln \det \mathbf{C}$ term is <i>not</i> a trivial constant when $\mathbf{C}$ has learnable hyperparameters — it penalises complexity and acts as an automatic Occam's razor, favouring simpler covariance structures that still explain the data.

<h2> 0.2 The Covariance Matrix</h2>

$\mathbf{C}$ is the covariance matrix of the noise. Its diagonal entries $C_{ii} = \sigma_i^2$ are the variances. The off-diagonal entries $C_{ij}$ (for $i \neq j$) are the covariances between the noise at different data points. Before, when dealing with independant noise, the off diagonals were zero, which meant that $\mathbf{C}^{-1}$ was $\text{diag}[1/\sigma_1^2, \ldots, 1/\sigma_N^2]$, which was represented by the denominator in the regular $\chi^2$ formula. This meant that you didn't need to think of the covariance matrix (or its inverse) at all.

If you want to deal with correlated noise, then you must consider $\mathbf{C}$. However, in general you do not know the off diagonal entries of $\mathbf{C}$. You also can't fit for the off-diagonal entries, as the number of entries to fit for grows as $\sim N^2/2$, which far outpaces the constraining power of the $N$ data points you have. So instead of fitting for each entry individually, you assume a *functional form* of your covariance. This means you assume that your covariance matrix has some structure that can be well approximated by some function with only a handful of parameters. This makes the problem tractable. 

These functions are known as *kernels*.

<h2> 0.3 Kernels</h2>

A kernel function takes two input locations and returns the covariance between them. We typically write the kernel function as:

$$ C_{ij} = k(x_i, x_j) $$

where $k$ is the kernal function. Kernel functions can be thought as us applying our prior knowledge about the data/noise to the covariance matrix, i.e. if the signal ought to be smooth, jagged, differentiable, or periodic, and on what scales. There are a number of conditions about what makes a valid kernel function, but to speak loosely, the most important condition is that whatever $\mathbf{C}$ matrix they make, it must be invertable. Speaking of invertibility....

<h2> 0.4 The bottleneck</h2>

Gaussian processes are very powerful, but they have one main drawback. This is that they scale terribly. The problem arises from the fact that we must calculate $\mathbf{C}^{-1}$. Matrix inversion, naively, scales as $\mathcal{O}(N^3)$. Some kernel functions and techniques can help mitigate this, but all of these are essentially doing linear algebra tricks to approximate or simplify the matrix inversion. 

<h2> 0.5 What is a Gaussian Process? </h2>

A Gaussian process is a prior over functions. Given a set of input locations, a GP defines a joint Gaussian distribution over the function values at those locations. That distribution is fully specified by two things: a <b>mean function</b> $\mu(x)$ and a <b>kernel function</b> $k(x_i, x_j)$:

$$f \sim \mathcal{GP}\big(\mu(x),\; k(x_i, x_j)\big)$$

The mean function is often set to zero (or a simple parametric model), leaving the kernel to do the heavy lifting. However, the mean function can be things like a transit model, or spectral templates. Conditioning this prior on observed data gives you a posterior distribution over functions.

-----------

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
 
<h1>§1. tinygp</h1>
 
In this tutorial, we will be using <code>tinygp</code>. It is not the only GP package, nor is it necessarily the best. The <code>tinygp</code> docs mention the following:

<ul>
<li><code>sklearn.gaussian_process</code></li>
<li><code>GPy</code></li>
<li><code>GPyTorch</code></li>
<li><code>george</code> [deprecated]</li>
<li><code>celerite</code></li>
<li><code>GPJax</code></li>
<li><code>jax-kern</code></li>
</ul>

The author of the docs states that for <code>GPyTorch</code>, it may be particularly useful for large datasets as it includes some scalable linear algebra magic. <code>celerite</code> is very fast for a specific kernel structure for one-dimensional data only, but can be a good choice. They also state that <code>GPJax</code> and <code>jax-kern</code> are interesting and promising, but may not be ready for public consumption. 

It should be noted that Gaussian Processes, and efficient algorithms for them, are an active field of study, so by the time you feel like messing with them for your own problems, another package might be a better choice for you.

<a href='https://tinygp.readthedocs.io/en/stable/tutorials.html'>There are a number of good tutorials for tinygp here.</a> 

<hr>

<h2>§1.1 Road Map</h2>

In this section, we will walk through part of a GP workflow on a toy problem: modelling the light curve of a spotted, rotating star. We will do this without any fitting Instead, we will generate synthetic data from a GP with known hyperparameters, and then see how well the GP can recover the underlying signal. This is just a sanity check / demonstration of the package.


The steps are:

<ol>
<li><b>Build a kernel</b> — define the covariance structure (§1.2)</li>
<li><b>Visualise the kernel</b> — what does the covariance function look like? (§1.2)</li>
<li><b>Create a GP object</b> — attach the kernel to observation coordinates and uncertainties (§1.3)</li>
<li><b>Draw from the prior</b> — sample functions <i>before</i> seeing data (§1.3)</li>
<li><b>Generate synthetic data</b> — one prior draw becomes our "truth", add noise to get observations (§1.3)</li>
<li><b>Condition on data</b> — compute the posterior: mean prediction and uncertainty (§1.3)</li>
<li><b>Draw posterior samples</b> — individual plausible light curves given the data (§1.3)</li>
</ol>

<img src="AigrainFig4.png" style="max-width: 100%; margin: 10px 0;">
    
**Figure 4 from Aigrain &amp; Foreman-Mackey (2023). Original caption:** *Schematic representation of a typical GPR workflow. Given a dataset (blue box) and some modelling choices (grey boxes, Section 3), the mathematical framework presented in Section 2 can be used (green boxes) to evaluate the likelihood and posterior distribution over the hyperparameters which can be optimised or sampled. Note that, for simplicity, we have assumed a zero mean function in this figure. We can also condition the GP on the data to predict observations at locations where we do not yet have observations*

<hr>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§1.2 Building the Kernel</h2>

<code>tinygp</code> provides a set of kernel objects in <code>tinygp.kernels</code>. The most commonly used ones are:

<br><br>

<table>
<tr><th>Kernel class</th><th>Formula</th><th>Parameters</th></tr>
<tr>
<td><code>ExpSquared(scale)</code></td>
<td>$k(r) = \exp(-r^2/2)$, where $r = |x - x'| / \ell$</td>
<td><code>scale</code> = lengthscale $\ell$</td>
</tr>
<tr>
<td><code>Matern32(scale)</code></td>
<td>$k(r) = (1 + \sqrt{3}\,r)\exp(-\sqrt{3}\,r)$</td>
<td><code>scale</code> = lengthscale $\ell$</td>
</tr>
<tr>
<td><code>Matern52(scale)</code></td>
<td>$k(r) = (1 + \sqrt{5}\,r + \frac{5}{3}r^2)\exp(-\sqrt{5}\,r)$</td>
<td><code>scale</code> = lengthscale $\ell$</td>
</tr>
<tr>
<td><code>ExpSineSquared(scale, gamma)</code></td>
<td>$k(r) = \exp(-\Gamma \sin^2 \pi r)$</td>
<td><code>scale</code> = period $P$, <code>gamma</code> = $\Gamma$</td>
</tr>
</table>

<br>

All stationary kernels have a <code>scale</code> parameter. For most kernels, this is the length scale $\ell$. For <code>ExpSineSquared</code>, the <code>scale</code> parameter is the period $P$. This is a confusing naming convention, so be aware of it.

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">Version note: <code>scale</code> vs <code>period</code></summary>
<div style="margin-top: 10px;">
In older versions of <code>tinygp</code> (v0.1.x), <code>ExpSineSquared</code> used a keyword argument called <code>period</code>. In newer versions, this was renamed to <code>scale</code> for consistency with the other stationary kernels. If you get an error like <code>TypeError: unexpected keyword argument 'scale'</code>, you are on an older version — use <code>period</code> instead. You can check your version with <code>print(tinygp.__version__)</code>.
</div>
</details>

Let's start with the most common kernel — the Squared Exponential — and see what it does.

</div>

In [ ]:
# A single kernel: Squared Exponential with lengthscale = 2.0
k_se = kernels.ExpSquared(scale=2.0)

# .evaluate() computes the covariance between a single pair of inputs
print(f"k(0.0, 0.0) = {k_se.evaluate(0.0, 0.0):.4f}")   # Zero separation --> 1.0
print(f"k(0.0, 1.0) = {k_se.evaluate(0.0, 1.0):.4f}")   # Separation < l --> strong
print(f"k(0.0, 5.0) = {k_se.evaluate(0.0, 5.0):.4f}")   # Separation > l --? weak

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

The Squared Exponential kernel has the form:

$$k_\text{SE}(x_i, x_j) = A^2 \exp\!\left(-\frac{(x_i - x_j)^2}{2\ell^2}\right)$$

In <code>tinygp</code>, you don't give the amplitude ($A$) directly. Instaed you multiply the kernel object by a scalar (see below). Above, $A = 1$. 

The key parameter is the length scale $\ell$. Points separated by less than $\ell$ are strongly correlated; points separated by much more than $\ell$ are essentially uncorrelated. Let's visualise this.

</div>

In [ ]:
# Visualise how the SE kernel varies with separation
separations = np.linspace(0, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: kernel as a function of separation for different lengthscales
ax = axes[0]
for ell in [0.5, 1.0, 2.0, 5.0]:
    k = kernels.ExpSquared(scale=ell)
    covariance = np.array([k.evaluate(0.0, dx) for dx in separations])
    ax.plot(separations, covariance, label=rf'$\ell = {ell}$', lw=2)

ax.set_xlabel(r'Separation $|x_i - x_j|$')
ax.set_ylabel(r'$k(x_i, x_j)$')
ax.set_title('SE kernel: effect of lengthscale')
ax.legend()
ax.set_ylim(-0.05, 1.05)

# Right: the covariance matrix for l = 2
ax = axes[1]
t_grid = jnp.linspace(0, 10, 50)
k_demo = kernels.ExpSquared(scale=2.0)
C = k_demo(t_grid, t_grid)  # This builds the full covariance matrix

im = ax.imshow(C, extent=[0, 10, 10, 0], cmap='viridis', aspect='auto')
ax.set_xlabel('$t_i$ (days)')
ax.set_ylabel('$t_j$ (days)')
ax.set_title(r'Covariance matrix $\mathbf{C}$, $\ell = 2$')
plt.colorbar(im, ax=ax, label=r'$k(t_i, t_j)$')

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

The left panel shows how the covariance falls off with separation. A longer lengthscale means points further apart are still correlated.

The right panel shows the covariance matrix $\mathbf{C}$ itself. This is the matrix from the recap. The diagonal is brightest (self-covariance = 1), and the off-diagonals decay as points get further apart. Note that we haven't added the measurement error to the diagonal.

<h3>Setting the amplitude</h3>

The amplitude (variance of the process) is controlled by multiplying the kernel by a scalar. If you want the standard deviation of the process to be $\sigma_f$, you multiply by $\sigma_f^2$:

</div>

In [ ]:
# If we want the GP to have std dev of 0.01 (1% flux variations):
sigma_f = 0.01
k_with_amp = sigma_f**2 * kernels.ExpSquared(scale=2.0)

# k(0,0) is now sigma_f^2, not 1
print(f"k(0, 0) = {k_with_amp.evaluate(0.0, 0.0):.6f}  (should be sigma_f^2 = {sigma_f**2:.6f})")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Combining kernels</h3>

Kernels can be added and multiplied to build more complex covariance structures:

<ul>
<li><b>Sum</b> $k = k_1 + k_2$: the signal has independent contributions from two processes. Useful for: long-term trend + short-term oscillation.</li>
<li><b>Product</b> $k = k_1 \times k_2$: the signal has <i>both</i> properties simultaneously.</li>
</ul>

For stellar rotation, we want a quasi-periodic kernel. A star with spots rotates, producing a periodic flux modulation. But the spots evolve so the periodic pattern changes over time.

$$k_\text{QP}(t_i, t_j) = \underbrace{\sigma_f^2}_{\text{amplitude}} \, \underbrace{\exp\left(-\Gamma \sin^2 \frac{\pi |t_i - t_j|}{P}\right)}_{\texttt{ExpSineSquared} \text{ (periodic)}} \underbrace{\exp\left(-\frac{(t_i - t_j)^2}{2\ell_e^2}\right)}_{\texttt{ExpSquared} \text{ (envelope)}}$$

The hyperparameters are:

<table>
<tr><th>Symbol</th><th>Name</th><th>Physical meaning</th><th>Value we'll use</th></tr>
<tr><td>$\sigma_f$</td><td>Amplitude</td><td>RMS of the flux variability</td><td>0.01 (1%)</td></tr>
<tr><td>$P$</td><td>Period</td><td>Stellar rotation period</td><td>5 days</td></tr>
<tr><td>$\ell_e$</td><td>Evolution timescale</td><td>How quickly spots change</td><td>15 days (~3 rotations)</td></tr>
<tr><td>$\Gamma$</td><td>Gamma</td><td>Smoothness within each period</td><td>1.0</td></tr>
</table>

Mess around with these hyperparameters below to see how they affect the curves.

</div>

In [ ]:
# Quasi-periodic kernel for a spotted, rotating star
P_rot      = 5.0    # Rotation period (days)
ell_e      = 15.0   # Spot evolution timescale (days)
sigma_f    = 0.005  # Flux variability amplitude (0.5%)
gamma_val  = 1.0    # Gamma (smoothness within each period)

k_periodic = kernels.ExpSineSquared(scale=P_rot, gamma=gamma_val)
k_envelope = kernels.ExpSquared(scale=ell_e)

# combine kerels
k_qp = sigma_f**2 * k_periodic * k_envelope

In [ ]:
separations = np.linspace(0, 50, 500)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# The three kernels to compare
kernel_list = [
    (sigma_f**2 * k_periodic,  'ExpSineSquared\n(strictly periodic)'),
    (sigma_f**2 * k_envelope,  'ExpSquared\n(decaying envelope)'),
    (k_qp,                     'Quasi-periodic\n(product of both)'),
]

for col, (k, name) in enumerate(kernel_list):
    # Top row: kernel as a function of separation
    ax = axes[0, col]
    covariance = np.array([k.evaluate(0.0, dx) for dx in separations])
    ax.plot(separations, covariance, 'k', lw=1.5)
    ax.set_xlabel(r'Separation $|t_i - t_j|$ (days)')
    if col == 0:
        ax.set_ylabel(r'$k(t_i, t_j)$')
    ax.set_title(name)
    ax.axhline(0, color='grey', ls=':', lw=0.5)

    # Bottom row: covariance matrix
    ax = axes[1, col]
    t_grid = jnp.linspace(0, 50, 100)
    C = k(t_grid, t_grid)
    im = ax.imshow(C, extent=[0, 50, 50, 0], cmap='viridis', aspect='auto')
    ax.set_xlabel('$t_i$ (days)')
    if col == 0:
        ax.set_ylabel('$t_j$ (days)')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<b>Left column (periodic):</b> The covariance oscillates forever with period $P = 5$ days. The covariance matrix has a repeating ridge pattern which means the GP "knows" that points separated by exactly $P$ should look similar, no matter how far apart they are in time.

<b>Centre column (SE envelope):</b> The covariance decays smoothly with separation. Points more than ~$\ell_e = 15$ days apart are essentially uncorrelated. There is no periodicity.

<b>Right column (quasi-periodic = product):</b> The covariance oscillates with period $P$, but the oscillations decay with timescale $\ell_e$. The covariance matrix shows periodic structure near the diagonal, which fades further away. This is what we want for starspots: the light curve looks periodic over a few rotations, but the pattern changes on longer timescales.

<hr>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§1.3 Prediction</h2>

We now have a kernel. The next step is to turn it into a <code>GaussianProcess</code> object — this is what actually does the computation. To create one, we need three things:

<ol>
<li>A kernel — we just built <code>k_qp</code>.</li>
<li>The input coordinates — the times at which we have (or will have) observations.</li>
<li>The diagonal noise — the measurement variances $\sigma_i^2$ at each point.</li>
</ol>

In code:

$$\underbrace{\texttt{GaussianProcess(kernel, t, diag=sigma**2)}}_{\text{This builds the covariance matrix } C_{ij} = k(t_i, t_j) + \sigma_i^2 \delta_{ij}}$$


<h3>Generating the observation schedule</h3>

Let's simulate a semi-realistic observing campaign. We will 100 days of observations, generally unevenly sampled, with a week-long gap in the middle (clouded out) and a few nights missing elsewhere.

</div>

In [ ]:
np.random.seed(42)

# Generate ~150 unevenly-spaced observation times over 100 days
t_all = np.sort(np.random.uniform(0, 100, 200))

# Remove a week-long gap (days 40-47) to simulate bad weather
mask = ~((t_all > 40) & (t_all < 47))

# Also remove a few scattered nights
mask &= ~((t_all > 15) & (t_all < 17))
mask &= ~((t_all > 72) & (t_all < 74))

t_obs = jnp.array(t_all[mask])
N = len(t_obs)

# Heteroscedastic uncertainties (different SNR on different nights)
sigma_obs = jnp.array(np.random.uniform(0.0005, 0.002, N))

print(f"Number of observations: {N}")
print(f"Time range: {float(t_obs[0]):.1f} – {float(t_obs[-1]):.1f} days")

fig, ax = plt.subplots(figsize=(12, 1.5))
ax.eventplot([np.array(t_obs)], lineoffsets=0, linelengths=0.5, color='k', lw=0.5)
ax.axvspan(40, 47, color='red', alpha=0.3, label='Week-long gap')
ax.axvspan(15, 17, color='salmon', alpha=0.3, label='(some of) Missing Nights')
ax.axvspan(72, 74, color='salmon', alpha=0.3)
ax.set_xlabel('Time (days)')
ax.set_yticks([])
ax.set_xlim([0,100])
ax.set_title('Observation schedule')
ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Creating the GP object</h3>

Now we pass the kernel, the observation times, and the measurement variances to <code>GaussianProcess</code>. This is the object that will do all the heavy lifting. Under the hood, it builds the covariance matrix:

$$C_{ij} = k_\text{QP}(t_i, t_j) + \sigma_i^2 \delta_{ij}$$

The first term comes from the kernel (the correlated structure we expect in the signal). The second term is the measurement noise. 

In code, the kernel provides $k_\text{QP}(t_i, t_j)$, and the <code>diag</code> argument provides $\sigma_i^2$:

</div>

In [ ]:
gp = GaussianProcess(k_qp, t_obs, diag=sigma_obs**2)
print(f"GP created with {N} observations")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Drawing from the prior</h3>

Before we show the GP any data, it already has beliefs about what functions look like (this is the prior). We can draw samples from it using <code>gp.sample(key)</code>.

*Note: JAX uses explicit random keys rather than a global seed. You create one with <code>jax.random.PRNGKey(seed)</code>. Different seeds give different draws.*

These prior samples are what the GP thinks a stellar light curve <i>could</i> look like, based only on the kernel, before seeing any data at all. Check: do they look like quasi-periodic stellar variability with a ~5-day period?

</div>

In [ ]:
t_prior = jnp.linspace(0, 100, 2000)
gp_prior = GaussianProcess(k_qp, t_prior, diag=1e-12 * jnp.ones(len(t_prior)))

fig, ax = plt.subplots(figsize=(12, 4))
for i in range(5):
    key = jax.random.PRNGKey(i)
    sample = gp_prior.sample(key)
    ax.plot(t_prior, sample+i*0.02, lw=1, alpha=0.7)

ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux (+offset)')
ax.set_title('Prior draws from the quasi-periodic GP (before seeing any data)')
ax.set_xlim([0,100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Generating synthetic data</h3>

We want a "true" light curve that we can compare against later. The strategy is:

<ol>
<li>Draw a smooth realisation from the GP prior on a <b>dense</b> time grid (2000 points). This is our ground truth.</li>
<li>Evaluate that truth at the sparse, gappy observation times, that we made above.</li>
<li>Add noise to get realistic observations.</li>
</ol>

<b>Step 1:</b> Draw the truth on a dense grid. To get a smooth curve we can plot, we evaluate the GP prior on a fine time grid of 2000 points. The tiny <code>diag = 1e-12</code> is a numerical stability trick. 

*Note that in reality, this "step" is unneccessary, as we should already have the data. We are just doing this so we can get a realistic looking lightcurve without having to write one down analytically.*

</div>

In [ ]:
# Step 1: Draw a smooth "true" signal on a dense grid
t_dense = jnp.linspace(0, 100, 2000)
gp_truth = GaussianProcess(k_qp, t_dense, diag=1e-12 * jnp.ones(len(t_dense)))

key_truth = jax.random.PRNGKey(42)
f_truth_dense = gp_truth.sample(key_truth)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_dense, f_truth_dense, 'tab:blue', lw=1)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_title('True Light Curve')
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<b>Step 2:</b> Evaluate the truth at the observation times. We have the truth on a dense grid, but our telescope only observed at the sparse times in <code>t_obs</code>. We use <code>gp.condition()</code> to interpolate, condition the dense noiseless GP on its own sample, and predict at <code>t_obs</code>. This gives us the exact value of the true signal at each observation time.

*Note: Again, this is an unorthodox way of getting the data, we will explain what the code is doing shortly.*

</div>

In [ ]:
# Step 2: Get the truth at the sparse observation times
_, cond_truth = gp_truth.condition(f_truth_dense, t_obs)
f_clean = cond_truth.loc

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_dense, f_truth_dense, 'tab:blue', lw=1, alpha=0.7, label='True signal (dense)')
ax.plot(t_obs, f_clean, 'k.', ms=3, label='True signal at observation times')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_title('Sampled Light Curve')
ax.legend(fontsize=9)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<b>Step 3:</b> Add noise. Each observation has its own uncertainty $\sigma_i$. We draw a noise realisation from $\mathcal{N}(0, \sigma_i)$ and add it to the clean signal. This is what the telescope would actually measure.

</div>

In [ ]:
# Step 3: Add noise to get the "observed" data
noise_realisation = sigma_obs * jax.random.normal(jax.random.PRNGKey(0), shape=(N,))
f_obs = f_clean + noise_realisation

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_dense, f_truth_dense, 'tab:blue', lw=1, alpha=0.7, label='True signal')
ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=4, alpha=0.7,
            label='Observations (noisy)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_title('Sampled Light Curve with Noise')
ax.legend(fontsize=9)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

This is our synthetic dataset. The true signal (blue) is known to us but hidden from the GP. The GP will only see the noisy black points. We know want to see if the GP can recover the blue curve, including in the gaps.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Conditioning</h3>

Up to this point, we have used <code>GaussianProcess</code> objects in two slightly unusual ways: once to draw a smooth "truth" from the prior, and once to interpolate that truth onto our observation times. These were tricks for generating synthetic data. Now we do the thing GPs are actually for.

We have noisy observations <code>f_obs</code> at times <code>t_obs</code>. We want to predict the underlying signal at a fine grid of times including inside the gaps where we have no data. This is GP regression.

---------
<h4> Maths Recap</h4>

In the pre-tutorial (§3.4), we derived this. The GP defines a joint Gaussian distribution over the observed values $\mathbf{y}$ and the predictions $\mathbf{f}_*$:

$$\begin{pmatrix} \mathbf{y} \\ \mathbf{f}_* \end{pmatrix} \sim \mathcal{N}\!\left( \mathbf{0},\; \begin{pmatrix} \mathbf{K}_{ff} + \sigma_n^2 \mathbf{I} & \mathbf{K}_{f*} \\ \mathbf{K}_{*f} & \mathbf{K}_{**} \end{pmatrix} \right)$$

where $\mathbf{K}_{ff}$ is the kernel evaluated at all pairs of observation times, $\mathbf{K}_{**}$ at all pairs of prediction times, and $\mathbf{K}_{f*}$, $\mathbf{K}_{*f}$ are the cross terms. Conditioning on the observed data $\mathbf{y}$ gives the posterior:

$$\mathbf{f}_* \mid \mathbf{y} \sim \mathcal{N}\!\big(\bar{\mathbf{f}}_*,\;\text{cov}(\mathbf{f}_*)\big)$$

$$\bar{\mathbf{f}}_* = \mathbf{K}_{*f}\,\big(\mathbf{K}_{ff} + \sigma_n^2 \mathbf{I}\big)^{-1}\,\mathbf{y}$$

$$\text{cov}(\mathbf{f}_*) = \mathbf{K}_{**} - \mathbf{K}_{*f}\,\big(\mathbf{K}_{ff} + \sigma_n^2 \mathbf{I}\big)^{-1}\,\mathbf{K}_{f*}$$

The posterior mean $\bar{\mathbf{f}}_*$ is a weighted combination of the observations. Points nearby in time (high covariance via $\mathbf{K}_{*f}$) get more weight. The posterior covariance starts as the prior covariance $\mathbf{K}_{**}$ and subtracts the information gained from the data so that wherever the data constrain the prediction, the uncertainty shrinks.

-------

You don't need to implement any of this. <code>tinygp</code> does it in one line:

`_, cond_gp = gp.condition(y, X_test)`

This takes two arguments:
<ol>
<li><code>y</code> — the observed data (our noisy flux values <code>f_obs</code>).</li>
<li><code>X_test</code> — the times at which we want predictions (a fine grid, including the gaps).</li>
</ol>

It returns two things:
<ol>
<li>The log marginal likelihood — we will use this later for hyperparameter learning. For now we ignore it (hence the <code>_</code>).</li>
<li>A new <code>GaussianProcess</code> object representing the posterior at the prediction times, with:
<ul>
<li><code>.loc</code> — the posterior mean $\bar{\mathbf{f}}_*$</li>
<li><code>.variance</code> — the diagonal of $\text{cov}(\mathbf{f}_*)$</li>
<li><code>.sample(key)</code> — draw a function from the posterior</li>
</ul>
</li>
</ol>

Let's do it:

</div>

In [ ]:
# Define a fine grid of times to predict at (including inside the gaps)
t_pred = jnp.linspace(0, 100, 1000)

# Condition the GP on the observed data
_, cond_gp = gp.condition(f_obs, t_pred)

# Extract the posterior mean and standard deviation
pred_mean = cond_gp.loc
pred_std  = jnp.sqrt(cond_gp.variance)

print(f"Posterior mean shape:     {pred_mean.shape}  (one value per prediction time)")
print(f"Posterior variance shape: {cond_gp.variance.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# Uncertainty band (±2 sigma)
ax.fill_between(t_pred, pred_mean - 2*pred_std, pred_mean + 2*pred_std,
                alpha=0.2, color='tab:blue', label=r'GP $\pm 2\sigma$')

# Posterior mean
ax.plot(t_pred, pred_mean, 'tab:blue', lw=1.5, label='GP posterior mean')

# True signal (unknown to the GP)
ax.plot(t_dense, f_truth_dense, 'tab:orange', lw=1, alpha=1, label='True signal', ls ='--')

# Observations
ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5)

ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.legend(fontsize=9)
ax.set_title('GP posterior: mean and uncertainty')
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Look at this carefully:

<ul>
<li><b>Where we have data</b>: the posterior mean (blue) tracks the true signal (orange) closely, and the uncertainty band is narrow. The GP is confident here as the data constrain the prediction.</li>
<li><b>In the week-long gap</b> (days 40–47): the uncertainty expands. The GP doesn't know what happened during the gap, so it says so.Yet, the periodic kernel "remembers" the rotation period and makes a prediction based on the pattern before and after the gap. The true (mostly) signal sits within the uncertainty band.</li>
<li><b>At the edges</b>: the uncertainty also expands as the GP reverts towards its prior (zero mean) where there is no data to anchor it.</li>
</ul>

The good thing about GPs is that the uncertainty is data-dependent. It is wide where you have little information and narrow where you have lots.

<h3>Posterior samples</h3>

The posterior mean is a single summary. To see the full range of functions consistent with the data, we draw posterior samples. Each sample is an individual plausible light curve. Where data is dense, the samples agree. In the gaps, they diverge somewhat.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for i in range(20):
    key = jax.random.PRNGKey(100 + i)
    sample = cond_gp.sample(key)
    ax.plot(t_pred, sample, lw=0.4, alpha=0.4, color='tab:blue')

ax.plot(t_dense, f_truth_dense, 'tab:orange', lw=1.5, alpha=0.7, label='True signal')
ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5,
            label='Observations')
ax.axvspan(40, 47, color='red', alpha=0.07)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.legend(fontsize=9)
ax.set_title('20 posterior samples: plausible light curves given the data')
ax.set_xlim([0,100])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
for i in range(80):
    key = jax.random.PRNGKey(200 + i)
    sample = cond_gp.sample(key)
    ax.plot(t_pred, sample, lw=0.4, alpha=0.4, color='tab:blue')

ax.plot(t_dense, f_truth_dense, 'tab:orange', lw=1.5, alpha=0.7, label='True signal')
ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5,
            label='Observations')
ax.axvspan(40, 47, color='red', alpha=0.07)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.legend(fontsize=9)
ax.set_title('20 posterior samples: plausible light curves given the data')
ax.set_xlim([30,55])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

The bottom panel is the key plot. Inside the gap, the posterior samples fan out (a small amount here) they agree on the approximate phase (the periodic kernel "knows" the rotation period) but disagree on the amplitude and exact shape (because the spots may have evolved while we weren't looking). 

This completes the core GP workflow with known hyperparameters. We told the GP the correct period, evolution timescale, and amplitude, and it recovered the signal including a reasonable prediction inside a week-long data gap.

In practice, you don't know the hyperparameters. The next section covers how to learn them from the data.

<h3>Putting it all together</h3>

The walkthrough above was deliberately slow — lots of intermediate plots and explanation. In practice, the entire GP regression workflow (for known hyperparameters) is just a few lines. Here it is in one cell:

</div>

In [ ]:
# ============================================================
# GP regression in practice (known hyperparameters)
# ============================================================

# --- 1. Define the kernel ---
kernel = (0.01**2
          * kernels.ExpSineSquared(scale=5.0, gamma=1.0)
          * kernels.ExpSquared(scale=15.0))

# --- 2. Build the GP (kernel + data coordinates + measurement noise) ---
gp = GaussianProcess(kernel, t_obs, diag=sigma_obs**2)

# --- 3. Condition on the data and predict on a fine grid ---
t_pred = jnp.linspace(0, 100, 1000)
_, cond_gp = gp.condition(f_obs, t_pred)

# --- 4. Extract the posterior mean and uncertainty ---
pred_mean = cond_gp.loc
pred_std  = jnp.sqrt(cond_gp.variance)

# --- 5. Plot ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(t_pred, pred_mean - 2*pred_std, pred_mean + 2*pred_std,
                alpha=0.2, color='tab:blue', label=r'GP $\pm 2\sigma$')
ax.plot(t_pred, pred_mean, 'tab:blue', lw=1.5, label='GP posterior mean')
ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_xlim([0,100])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§2. Hyperparameter Learning</h1>

Everything above assumed we <i>knew</i> the correct hyperparameters. We told the GP the rotation period, the spot evolution timescale, and the amplitude. In practice, you may not know these. You can, however, learn them from the data.

<h2>§2.1 The Log Marginal Likelihood</h2>

From the pre-tutorial (§4), the log marginal likelihood of a GP is:

$$\ln p(\mathbf{y} \mid \boldsymbol{\phi}) = -\frac{1}{2} \left[ \underbrace{\mathbf{y}^T \mathbf{C}_\phi^{-1} \mathbf{y}}_{\text{data fit}} + \underbrace{\ln \det \mathbf{C}_\phi}_{\text{complexity}} + \underbrace{N \ln 2\pi}_{\text{constant}} \right]$$

where $\mathbf{C}_\phi = \mathbf{K}_\phi + \text{diag}(\boldsymbol{\sigma}^2)$ and $\boldsymbol{\phi}$ are the kernel hyperparameters.

In <code>tinygp</code>, this is computed by <code>gp.log_probability(y)</code>. 

<h2>§2.2 Point Estimates vs. Full Inference</h2>

There are two approaches to learning hyperparameters:

<ol>
<li><b>Optimisation</b>: find the $\boldsymbol{\phi}$ that maximises $\ln p(\mathbf{y} \mid \boldsymbol{\phi})$. Fast, but gives a single point estimate with no uncertainty.</li>
<li><b>MCMC</b>: sample from $p(\boldsymbol{\phi} \mid \mathbf{y}) \propto p(\mathbf{y} \mid \boldsymbol{\phi})\, p(\boldsymbol{\phi})$. Slower, but gives the full posterior over hyperparameters. Uncertainties in the hyperparameters propagate into the predictive distribution.</li>
</ol>

Let's do both. First, a quick optimisation to get a point estimate and sanity-check the model. Then MCMC for the full posterior.

<h3>Optimisation with JAX + scipy</h3>

Since <code>tinygp</code> is built on JAX, we can use <code>jax.grad</code> to compute the gradient of the log marginal likelihood and pass it to <code>scipy.optimize.minimize</code>. This is fast and gives a good starting point.

The workflow is:
<ol>
<li>Write a function that takes a vector of log-hyperparameters and returns the <i>negative</i> log marginal likelihood (negative because <code>scipy.minimize</code> minimises, but we want to maximise).</li>
<li>Use <code>jax.value_and_grad</code> to create a function that returns both the value and its gradient in one call.</li>
<li>Wrap it so <code>scipy</code> gets plain Python floats (it doesn't understand JAX arrays).</li>
</ol>

</div>

In [ ]:
# Step 1
@jax.jit
def neg_log_ml(params): # Input: a vector of LOG-hyperparameters (so they're unbounded)
    # extract parameters
    log_amp, log_P, log_gamma, log_ell_e = params

    # Create kernel using parameters
    k = jnp.exp(log_amp)**2 * (
        kernels.ExpSineSquared(scale=jnp.exp(log_P), gamma=jnp.exp(log_gamma))
        * kernels.ExpSquared(scale=jnp.exp(log_ell_e))
    )
    # Create GP object
    gp = GaussianProcess(k, t_obs, diag=sigma_obs**2)
    return -gp.log_probability(f_obs) # Output: the NEGATIVE log marginal likelihood (because scipy minimises).

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">What does <code>@jax.jit</code> do?</summary>
<div style="margin-top: 10px;">

The <code>@jax.jit</code> is there to make this function much faster. It traces the function once on the first call, builds a computational graph, and compiles it to optimised machine code via XLA (the same compiler backend that TensorFlow uses). Subsequent calls skip the Python interpreter entirely and run the compiled version. For our GP log marginal likelihood — which involves building a kernel matrix, computing a Cholesky decomposition, and doing a bunch of linear algebra — this is dramatically faster than pure Python/NumPy because XLA can fuse operations, eliminate intermediate arrays, and exploit hardware-specific optimisations. The trade-off is that the first call is slow (compilation overhead), but every subsequent call is fast. However, as this is going to get called every time we test some hyperparameters, it is worth doing.

</div>
</details>

</div>

In [ ]:
# Step 2: jax.value_and_grad returns BOTH the function value AND its
# gradient (via automatic differentiation) in a single call.

neg_log_ml_and_grad = jax.jit(jax.value_and_grad(neg_log_ml))

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">What does <code>jax.value_and_grad</code> do?</summary>
<div style="margin-top: 10px;">

<code>jax.value_and_grad(f)</code> takes a function <code>f(x) → y</code> and returns a new function that computes <b>both</b> <code>f(x)</code> and <code>∂f/∂x</code> in a single call. The gradient is computed via <b>automatic differentiation</b> — JAX walks through the computational graph of <code>f</code> and applies the chain rule symbolically, via magic, presumably.

This is what makes the JAX + tinygp combination powerful: our <code>neg_log_ml</code> function involves building a kernel matrix, Cholesky-decomposing it, and solving a linear system. JAX differentiates through all of that automatically. The gradient tells <code>scipy</code> which direction in hyperparameter space improves the fit, so L-BFGS-B converges in tens of steps rather than hundreds, as `scipy` doesn't need to approximate the gradient via finite difference.

We wrap it in another <code>jax.jit</code> so the combined value+gradient computation is also jit compiled.

</div>
</details>

In [ ]:
# Step 3: scipy expects plain Python floats, not JAX arrays.
# This thin wrapper does the conversion.
def objective(params):
    val, grad = neg_log_ml_and_grad(jnp.array(params))
    return float(val), [float(g) for g in grad]

In [ ]:
%%time
# Step 4: Optimise. jac=True tells scipy that our function returns
# (value, gradient) together, so it doesn't call them separately.
x0 = [jnp.log(0.01), jnp.log(5.0), 0.0, jnp.log(15.0)]
result = scipy_minimize(objective, x0, jac=True, method='L-BFGS-B')

# Convert back from log-space
amp_opt, P_opt, gamma_opt, ell_e_opt = jnp.exp(jnp.array(result.x))
print(f"Optimised hyperparameters:")
print(f"  Amplitude     = {amp_opt:.4f}")
print(f"  Period        = {P_opt:.2f} days")
print(f"  Gamma         = {gamma_opt:.2f}")
print(f"  Evolution ℓ_e = {ell_e_opt:.1f} days")
print(f"  Log ML        = {-result.fun:.1f}")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

This gives a point estimate in seconds. But optimisation it has two problems:

<ol>
<li><b>No uncertainties.</b> You get a single "best" set of hyperparameters but no sense of how constrained they are. If the marginal likelihood surface is broad, the optimum tells you very little.</li>
<li><b>Local optima.</b> The marginal likelihood can be multimodal. The optimiser finds <i>a</i> mode, not necessarily the global one.
</ol>

For astronomy, where we care about propagating uncertainties, MCMC (or some other sampling algorithm) is almost always preferable. But the optimisation result is useful as a sanity check and as an initialisation for the sampler.

<h2>§2.3 The NumPyro Model</h2>

The pattern for GP hyperparameter inference in NumPyro is:

<ol>
<li>Define priors on each hyperparameter.</li>
<li>Build the <code>tinygp</code> kernel and GP object from those parameters.</li>
<li>Use <code>numpyro.factor()</code> to add the GP log marginal likelihood to the model's log-density.</li>
</ol>

In a standard NumPyro model, you add a likelihood via <code>numpyro.sample("obs", dist.Normal(mu, sigma), obs=y)</code>. You give NumPyro a distribution and the data, and it evaluates the log-probability internally.

But here, <code>tinygp</code> already computes the full multivariate normal log-likelihood inside <code>gp.log_probability(y)</code> So instead, we use <code>numpyro.factor("gp", gp.log_probability(y))</code>, which says: "I've computed this log-probability term myself, add it to the model's total log-density." (Log Density is the unnormalised posterior, btw) NumPyro takes the scalar and includes it when computing gradients for NUTS. 

Here is a worked example using the synthetic data from §1. If some of the below code is confusing, look back at the MCMC tutorial.

</div>

In [ ]:
def gp_model(t, y, yerr):
    """
    NumPyro model for GP hyperparameter inference.
    Quasi-periodic kernel: amplitude * ExpSineSquared * ExpSquared
    """
    # --- Priors ---
    # Amplitude of the GP (std dev of the process)
    log_amp = numpyro.sample("log_amp", dist.Normal(jnp.log(0.01), 1.0))
    amp = jnp.exp(log_amp)
    
    # Rotation period (days)
    log_P = numpyro.sample("log_P", dist.Normal(jnp.log(5.0), 0.5))
    P = jnp.exp(log_P)
    
    # Gamma in ExpSineSquared
    log_gamma = numpyro.sample("log_gamma", dist.Normal(0.0, 1.0))
    gamma = jnp.exp(log_gamma)
    
    # Spot evolution timescale (days)
    log_ell_e = numpyro.sample("log_ell_e", dist.Normal(jnp.log(15.0), 1.0))
    ell_e = jnp.exp(log_ell_e)
    
    # --- Build kernel ---
    kernel = amp**2 * kernels.ExpSineSquared(scale=P, gamma=gamma) * kernels.ExpSquared(scale=ell_e)
    
    # --- Build GP and evaluate log marginal likelihood ---
    gp = GaussianProcess(kernel, t, diag=yerr**2)
    numpyro.factor("gp", gp.log_probability(y))
    
    # --- Store the GP object for prediction later ---
    numpyro.deterministic("pred_mean", gp.loc)  # just the prior mean (zero) — placeholder

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

A few things to note:

<ul>
<li>We sample in log-space for all positive parameters. This keeps them positive and gives NUTS an unbounded space to explore. The priors on the log-parameters are mildly informative &mdash; centred near reasonable values with enough width to explore.</li>
<li><code>numpyro.factor("gp", ...)</code> adds the GP log marginal likelihood directly to the model's log-density.</li>
<li>The <code>diag=yerr**2</code> argument passes the <i>per-point</i> noise variances (heteroscedastic noise), not a scalar.</li>
</ul>

The below cell runs the NUTS MCMC algorithm. It may initially give a long estimate for time till completion, but on my machine it speeds up significantly after 100-200 steps. 

</div>

In [ ]:
# Run MCMC
sampler = NUTS(gp_model, dense_mass=True)
mcmc = MCMC(sampler, num_warmup=250, num_samples=750, num_chains=1)
mcmc.run(random.PRNGKey(0), jnp.array(t_obs), jnp.array(f_obs), jnp.array(sigma_obs))

mcmc.print_summary(exclude_deterministic=True)

# Check for divergences
n_div = mcmc.get_extra_fields()['diverging'].sum()
print(f"\nDivergences: {n_div}")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

If you have done everything correctly, you should have an `r_hat = 1.00` for all the parameters, and several hundred `n_eff`.

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">Aside: Why <code>dense_mass=True</code>?</summary>
<div style="margin-top: 10px;">

GP hyperparameters are often correlated in the posterior (e.g., amplitude and length scale trade off). By default, NUTS uses a diagonal mass matrix, which assumes the parameters are uncorrelated. <code>dense_mass=True</code> lets NUTS learn the correlations during warmup, which can dramatically improve sampling efficiency for GP models. Try turning it off and comparing the ESS.

</div>
</details>

We can also plot the trace plots and chains, as before:

</div>

In [ ]:
# Trace plots and corner plot
idata = az.from_numpyro(mcmc)
az.plot_trace(idata, var_names=["log_amp", "log_P", "log_gamma", "log_ell_e"], figsize=(12, 8))
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§2.4 Posterior Predictive Distribution</h2>

Now we have posterior samples of the hyperparameters. For each sample, we can build a GP, condition on the data, and predict. The spread across these predictions gives us the posterior predictive distribution which should represent the uncertainty from both the GP conditioning <i>and</i> the hyperparameter uncertainty.

</div>

In [ ]:
# Posterior predictive: draw GP predictions for each hyperparameter sample
samples = mcmc.get_samples()
n_show = 50  # number of posterior samples to plot
t_pred = jnp.linspace(0, 100, 500)

fig, ax = plt.subplots(figsize=(12, 4))

for i in range(n_show):
    # Extract hyperparameters for this sample
    amp    = jnp.exp(samples["log_amp"][i])
    P      = jnp.exp(samples["log_P"][i])
    gamma  = jnp.exp(samples["log_gamma"][i])
    ell_e  = jnp.exp(samples["log_ell_e"][i])
    
    # Build kernel and GP with these hyperparameters
    k = amp**2 * kernels.ExpSineSquared(scale=P, gamma=gamma) * kernels.ExpSquared(scale=ell_e)
    gp_i = GaussianProcess(k, t_obs, diag=sigma_obs**2)
    _, cond_i = gp_i.condition(f_obs, t_pred)
    
    ax.plot(t_pred, cond_i.loc, lw=0.3, alpha=0.3, color='tab:blue')

ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5)
ax.plot(t_dense, f_truth_dense, 'tab:orange', lw=1.5, alpha=0.7, ls='--', label='True signal')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_title('Posterior predictive: 50 GP means drawn from the hyperparameter posterior')
ax.legend(fontsize=9)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§3. Exercises</h1>

The following exercises are designed to be worked through in order. Each builds on the previous one. Skeleton code is provided &mdash; fill in the <code>TODO</code> sections. Hints and solutions are in collapsible boxes.

<h2>§3.1 Exercise: Recover the Rotation Period</h2>

Using the MCMC results from §2, make a histogram of the posterior samples of the rotation period $P$. Does the posterior recover the true value ($P = 5.0$ days)?

</div>

In [ ]:
############################################################
# TODO: Plot a histogram of the posterior rotation period  #
# Hint: the samples are in log-space (samples["log_P"])    #
# The true value is P = 5.0 days                           #
############################################################

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
P_samples = np.exp(np.array(samples["log_P"]))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(P_samples, bins=40, density=True, alpha=0.7, color='tab:blue', edgecolor='white')
ax.axvline(5.0, color='tab:orange', ls='--', lw=2, label=f'True P = 5.0 d')
ax.axvline(np.median(P_samples), color='tab:red', ls='-', lw=2, 
           label=f'Median = {np.median(P_samples):.2f} d')
ax.set_xlabel('Rotation period (days)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print(f"P = {np.median(P_samples):.3f} +/- {np.std(P_samples):.3f} days")
```
</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§3.2 Exercise: What Happens with the Wrong Kernel?</h2>

The data was generated with a quasi-periodic kernel. What happens if we fit it with a plain Squared Exponential kernel instead? Reminder that this kernel has no periodic component.

Write a new NumPyro model using only an SE kernel (amplitude + length scale), run MCMC, and compare the posterior predictive to the quasi-periodic result.

</div>

In [ ]:
def gp_model_se(t, y, yerr):
    """
    NumPyro model with a WRONG kernel: plain Squared Exponential.
    """
    #######################################
    # TODO: Define priors for log_amp and log_ell
    #######################################
    log_amp = 
    log_ell = 
    
    amp = jnp.exp(log_amp)
    ell = jnp.exp(log_ell)
    
    #######################################
    # TODO: Build the SE kernel and GP
    # kernel = amp**2 * kernels.ExpSquared(scale=ell)
    #######################################
    kernel = 
    gp = GaussianProcess(kernel, t, diag=yerr**2)
    
    numpyro.factor("gp", gp.log_probability(y))

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

The SE kernel has only two hyperparameters: amplitude and length scale. Use similar priors to the QP model but without the period or gamma parameters. The model structure is otherwise identical.

</div>
</details>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
def gp_model_se(t, y, yerr):
    log_amp = numpyro.sample("log_amp", dist.Normal(jnp.log(0.01), 1.0))
    log_ell = numpyro.sample("log_ell", dist.Normal(jnp.log(5.0), 1.0))
    
    amp = jnp.exp(log_amp)
    ell = jnp.exp(log_ell)
    
    kernel = amp**2 * kernels.ExpSquared(scale=ell)
    gp = GaussianProcess(kernel, t, diag=yerr**2)
    numpyro.factor("gp", gp.log_probability(y))

sampler_se = NUTS(gp_model_se, dense_mass=True)
mcmc_se = MCMC(sampler_se, num_warmup=500, num_samples=1500, num_chains=4)
mcmc_se.run(random.PRNGKey(1), jnp.array(t_obs), jnp.array(f_obs), jnp.array(sigma_obs))
mcmc_se.print_summary()

# Posterior predictive
samples_se = mcmc_se.get_samples()
fig, ax = plt.subplots(figsize=(12, 4))
for i in range(50):
    amp = jnp.exp(samples_se["log_amp"][i])
    ell = jnp.exp(samples_se["log_ell"][i])
    k = amp**2 * kernels.ExpSquared(scale=ell)
    gp_i = GaussianProcess(k, t_obs, diag=sigma_obs**2)
    _, cond_i = gp_i.condition(f_obs, t_pred)
    ax.plot(t_pred, cond_i.loc, lw=0.3, alpha=0.3, color='tab:blue')

ax.errorbar(np.array(t_obs), np.array(f_obs), yerr=np.array(sigma_obs),
            fmt='.', color='k', ecolor='grey', capsize=0, ms=3, alpha=0.5)
ax.plot(t_dense, f_truth_dense, 'tab:orange', lw=1.5, alpha=0.7, ls='--', label='True signal')
ax.axvspan(40, 47, color='red', alpha=0.07)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
ax.set_title('Wrong kernel (SE only): posterior predictive')
ax.legend(fontsize=9)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()
```

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<b>Question:</b> How does the SE posterior predictive differ from the QP one, particularly in the data gap?

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

The SE kernel has no concept of periodicity. In the data gap, it reverts to the prior mean (zero) with expanding uncertainty, like a smooth interpolation. The QP kernel "knows" the rotation period and can predict the phase of the oscillation inside the gap, even though it wasn't observed there. The SE kernel cannot do this. This is why kernel choice matters as the kernel encodes your physical assumptions about the signal.

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§3.3 Exercise: Model Comparison via the Log Marginal Likelihood</h2>

We now have two models: QP and SE. Which one does the data prefer? 

The simplest approach: evaluate the log marginal likelihood at the <i>maximum a posteriori</i> (MAP) hyperparameters for each model. The model with the higher log marginal likelihood fits the data better (subject to the caveat that this ignores the Occam factor from the prior volume. For a proper comparison, you'd compute the Bayesian evidence via nested sampling, which you've already done in a previous tutorial). If you want to try this, you can wrap each GP model in a <code>dynesty</code> likelihood function and compute $\ln \mathcal{Z}$ for each — the difference gives you the log Bayes factor. The marginal likelihood comparison here is a fast approximation that skips the prior volume correction.

For a quick comparison, compute the log marginal likelihood at the median posterior hyperparameters for each model.

</div>

In [ ]:
##################################################################
# TODO: Compute the log marginal likelihood for each model       #
# at the median posterior hyperparameters                        #
#                                                                #
# For each model:                                                #
# 1. Extract median hyperparameters from the posterior samples   #
# 2. Build the kernel and GP                                     #
# 3. Call gp.log_probability(f_obs)                              #
#                                                                #
# Compare the two values.                                        #
##################################################################

# QP model
log_ml_qp = ...

# SE model  
log_ml_se = ...

print(f"Log marginal likelihood (QP): {log_ml_qp:.1f}")
print(f"Log marginal likelihood (SE): {log_ml_se:.1f}")
print(f"Difference (QP - SE): {log_ml_qp - log_ml_se:.1f}")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
# QP model: median hyperparameters
amp_qp   = jnp.exp(jnp.median(samples["log_amp"]))
P_qp     = jnp.exp(jnp.median(samples["log_P"]))
gamma_qp = jnp.exp(jnp.median(samples["log_gamma"]))
ell_e_qp = jnp.exp(jnp.median(samples["log_ell_e"]))

k_qp_med = amp_qp**2 * kernels.ExpSineSquared(scale=P_qp, gamma=gamma_qp) * kernels.ExpSquared(scale=ell_e_qp)
gp_qp = GaussianProcess(k_qp_med, t_obs, diag=sigma_obs**2)
log_ml_qp = gp_qp.log_probability(f_obs)

# SE model: median hyperparameters
amp_se = jnp.exp(jnp.median(samples_se["log_amp"]))
ell_se = jnp.exp(jnp.median(samples_se["log_ell"]))

k_se_med = amp_se**2 * kernels.ExpSquared(scale=ell_se)
gp_se = GaussianProcess(k_se_med, t_obs, diag=sigma_obs**2)
log_ml_se = gp_se.log_probability(f_obs)

print(f"Log marginal likelihood (QP): {log_ml_qp:.1f}")
print(f"Log marginal likelihood (SE): {log_ml_se:.1f}")
print(f"Difference (QP - SE):         {log_ml_qp - log_ml_se:.1f}")
```

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

You should see a difference of ~70 in log marginal likelihood, which is massive. Since these are logarithms, the actual likelihood ratio is $e^{70} \approx 10^{30}$ i.e. the data are $10^{30}$ times more probable under the QP kernel than the SE kernel. For context, a log Bayes factor of 5 is conventionally considered "strong" evidence and 10 is "decisive" (on the Jeffreys scale).

This makes physical sense: the data <i>are</i> periodic, and the QP kernel knows this while the SE kernel does not. The SE kernel wastes its flexibility trying to approximate periodicity with a smooth, non-periodic function, and it does a poor job.

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">This comparison is only approximate...</summary>
<div style="margin-top: 10px;">

We evaluated the log marginal likelihood at the <i>median</i> posterior hyperparameters, not the MAP, and we didn't account for the different prior volumes of the two models (the QP model has 4 hyperparameters, the SE model has 2, so the QP model pays a larger Occam penalty). A proper Bayesian model comparison would compute the evidence $\mathcal{Z} = \int p(\mathbf{y} \mid \boldsymbol{\phi})\, p(\boldsymbol{\phi})\, d\boldsymbol{\phi}$ for each model via nested sampling, as in the nested sampling tutorial. But when the log marginal likelihood difference is this large, the Occam correction is negligible — the QP kernel wins decisively.

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§3.4 Exercise: Transit Fitting with Correlated Noise</h2>

So far, our GP has had a zero mean function: we're modelling the data as pure GP. In practice, you have a physical model (the "mean function") plus correlated noise (the GP).

We will demonstrate this by fitting a transit light curve in the presence of instrumental systematics. The transit is the signal you care about; the systematics are correlated noise that must be modelled simultaneously.

We'll generate a single transit observation with smooth instrumental trends (from thermal cycling, pointing drift, etc.). These systematics are <i>not</i> drawn from any GP kernel. They're deterministic functions of time, just like real instrumental effects. The GP has to be flexible enough to capture them anyway.

<h3>The transit model</h3>

We use a simple analytic limb-darkened transit. The key parameters are:
<ul>
<li>$T_0$: mid-transit time</li>
<li>$R_p/R_\star$: planet-to-star radius ratio (sets the transit depth: depth $\approx (R_p/R_\star)^2$)</li>
<li>$a/R_\star$: scaled semi-major axis (sets the transit duration together with inclination)</li>
<li>$i$: orbital inclination</li>
<li>$u_1, u_2$: limb darkening coefficients</li>
</ul>

For simplicity, we'll fix $a/R_\star$, $i$, $u_1$, and $u_2$ to known values and only fit $T_0$ and $R_p/R_\star$.

</div>

In [ ]:
def transit_model_jax(t, T0, rp_rs, a_rs=8.0, inc_deg=87.0, u1=0.3, u2=0.1):
    """JAX-compatible limb-darkened transit model.
    
    Uses a sigmoid approximation for ingress/egress to keep the function
    differentiable everywhere (required by NUTS).
    """
    inc = jnp.radians(inc_deg)
    z = a_rs * jnp.sqrt(jnp.sin(2 * jnp.pi * (t - T0))**2 + 
                         (jnp.cos(inc) * jnp.cos(2 * jnp.pi * (t - T0)))**2)
    p = rp_rs
    norm = 1 - u1/3 - u2/6
    mu = jnp.sqrt(jnp.clip(1 - z**2, 1e-12))
    limb = 1 - u1 * (1 - mu) - u2 * (1 - mu)**2
    in_transit = jax.nn.sigmoid(50.0 * (1 - p - z))
    flux = 1 - p**2 * limb / norm * in_transit
    return flux

In [ ]:
# ============================================================
# Generate a synthetic transit observation with instrumental systematics
# ============================================================
np.random.seed(161)

# --- Observation times: ~0.3 days centred on transit ---
N_transit = 100
t_tr = np.sort(np.random.uniform(-0.15, 0.15, N_transit))

# --- True transit parameters ---
T0_true    = 0.0      # mid-transit time (days)
rp_rs_true = 0.12     # Rp/Rs = 0.12 → depth ≈ 1.4%

# True transit
flux_transit = np.array(transit_model_jax(jnp.array(t_tr), T0_true, rp_rs_true))

# --- Instrumental systematics (NOT drawn from a GP kernel) ---
# These mimic real effects: thermal breathing (sinusoidal),
# detector settling (exponential ramp), and a linear drift.
systematic = (
    0.003 * np.sin(2 * np.pi * t_tr / 0.08)           # thermal breathing (~HST orbit)
    + 0.004 * np.exp(-((t_tr + 0.2) / 0.05))          # exponential ramp at start
    - 0.002 * t_tr                                     # linear drift
    + 0.002 * np.sin(2 * np.pi * t_tr / 0.25 + 1.0)   # slower oscillation
)

# --- White noise ---
sigma_tr = 0.001 * np.ones(N_transit)  # 1 mmag per point
noise = sigma_tr * np.random.randn(N_transit)

# --- Total observed flux ---
f_obs_tr = flux_transit + systematic + noise

# --- JAX versions ---
t_tr_jax     = jnp.array(t_tr)
f_obs_tr_jax = jnp.array(f_obs_tr)
sigma_tr_jax = jnp.array(sigma_tr)


# Quick check
print("transit_model_jax output shape:", transit_model_jax(jnp.array(t_tr), 0.0, 0.1).shape)

In [ ]:
# Visualise the data and its components
t_fine_plot = np.linspace(-0.15, 0.15, 1000)
flux_transit_fine = np.array(transit_model_jax(jnp.array(t_fine_plot), T0_true, rp_rs_true))
systematic_fine = (
    0.003 * np.sin(2 * np.pi * t_fine_plot / 0.08)
    + 0.004 * np.exp(-((t_fine_plot + 0.2) / 0.05))
    - 0.002 * t_fine_plot
    + 0.002 * np.sin(2 * np.pi * t_fine_plot / 0.25 + 1.0)
)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].errorbar(t_tr, f_obs_tr, yerr=sigma_tr, fmt='.', color='k', 
                 ecolor='grey', capsize=0, ms=3, alpha=0.7)
axes[0].plot(t_fine_plot, flux_transit_fine + systematic_fine, 'black', lw=1.5, alpha=0.7, 
             label='Transit + systematics (truth)', ls = '--')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Observed flux')
axes[0].set_title('What we observe: transit + systematics + noise')

axes[1].plot(t_fine_plot, flux_transit_fine, 'tab:blue', lw=2, label='Transit')
axes[1].plot(t_fine_plot, 1 + systematic_fine, 'tab:red', lw=1.5, alpha=0.7, label='Systematics')
axes[1].legend(fontsize=9)
axes[1].set_ylabel('Components')
axes[1].set_title('Truth (unknown to the observer)')

axes[2].errorbar(t_tr, f_obs_tr - systematic, yerr=sigma_tr, fmt='.', color='k',
                 ecolor='grey', capsize=0, ms=3, alpha=0.7)
axes[2].plot(t_fine_plot, flux_transit_fine, 'tab:blue', lw=2)
axes[2].set_ylabel('Flux')
axes[2].set_xlabel('Time (days)')
axes[2].set_title('If you could subtract systematics perfectly...')

plt.tight_layout()
plt.show()

print(f"Transit depth = {(1 - flux_transit.min())*100:.2f}%")
print(f"Systematics peak-to-peak = {(systematic.max() - systematic.min())*100:.2f}%")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

The systematics are comparable in amplitude to the transit depth. If you ignore them, your transit parameters will be biased.

<b>Task:</b> Fit this light curve two ways:
<ol>
<li><b>Without GP:</b> fit the transit model assuming white noise only. This ignores the systematics entirely.</li>
<li><b>With GP:</b> fit the transit model as the mean function, with a GP modelling the systematics. Use a Matérn-3/2 kernel (smooth but not infinitely differentiable — a good default for instrumental trends).</li>
</ol>

Compare the posteriors on $T_0$ and $R_p/R_\star$. The without-GP fit should give overconfident, biased posteriors. The with-GP fit should give wider, honest posteriors that encompass the truth.

<h3>A note on the mean function</h3>

When we pass <code>mean=flux_model</code> to <code>GaussianProcess</code> (or equivalently compute residuals <code>y - flux_model</code> and model those with a zero-mean GP), the GP models the <i>residuals</i> $\mathbf{y} - \mathbf{m}(\boldsymbol{\theta})$ as a draw from a zero-mean Gaussian process. This is exactly the "GP on the residuals" pattern: the transit is the deterministic signal, the GP captures everything else that is correlated.

</div>

In [ ]:
# ===========================================================
# Fit 1: Transit model only (no GP — assumes white noise)
# ===========================================================

def no_gp_transit_model(t, y, yerr):
    """Transit-only model: assumes white noise."""
    #######################################
    # TODO: Priors for T0 and log_rp_rs  #
    # Suggested:                          #
    #   T0:        Normal(0, 0.05)        #
    #   log_rp_rs: Normal(log(0.1), 0.3)  #
    #######################################
    T0        = ...
    log_rp_rs = ...
    rp_rs     = jnp.exp(log_rp_rs)
    
    flux_model = transit_model_jax(t, T0, rp_rs)
    numpyro.sample("obs", dist.Normal(flux_model, yerr), obs=y)

In [ ]:
# ===========================================================
# Fit 2: Transit + GP (Matérn-3/2 for systematics)
# ===========================================================

def gp_transit_model(t, y, yerr):
    """Transit + GP model: transit as mean function, GP for systematics."""
    # --- Transit parameters (same priors as above) ---
    T0        = ...
    log_rp_rs = ...
    rp_rs     = jnp.exp(log_rp_rs)
    
    # Residuals: data minus transit model
    residuals = y - transit_model_jax(t, T0, rp_rs)
    
    # --- GP hyperparameters for systematics ---
    ##########################################################################
    # TODO: Priors for GP amplitude and length scale                         #
    # Suggested:                                                             #
    #   log_amp_gp: Normal(log(0.003), 1.0)  — systematics are ~mmag level  #
    #   log_ell_gp: Normal(log(0.08), 0.5)   — varies on ~hour timescales   #
    ##########################################################################
    log_amp_gp = ...
    log_ell_gp = ...
    amp_gp = jnp.exp(log_amp_gp)
    ell_gp = jnp.exp(log_ell_gp)
    
    ##################################################################
    # TODO: Build Matern32 kernel, create ZERO-MEAN GP on residuals, #
    #       call numpyro.factor with the GP log probability           #
    ##################################################################
    ...

In [ ]:
# ===========================================================
# Run both fits
# ===========================================================

init_vals = {"T0": jnp.array(0.0), "log_rp_rs": jnp.array(jnp.log(0.1))}

# --- No GP ---
mcmc_no_gp = MCMC(
    NUTS(no_gp_transit_model, init_strategy=init_to_value(values=init_vals)),
    num_warmup=200, num_samples=1000, num_chains=1
)
mcmc_no_gp.run(random.PRNGKey(5), t_tr_jax, f_obs_tr_jax, sigma_tr_jax)

# --- With GP (use dense_mass=True for correlated parameters) ---
# takes around 2-3 mins
mcmc_gp = MCMC(
    NUTS(gp_transit_model, dense_mass=True, init_strategy=init_to_value(values=init_vals)),
    num_warmup=200, num_samples=1000, num_chains=1
)
mcmc_gp.run(random.PRNGKey(6), t_tr_jax, f_obs_tr_jax, sigma_tr_jax)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Checklist</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
<ol>
<li>Define <code>no_gp_transit_model(t, y, yerr)</code>: sample <code>T0</code> and <code>log_rp_rs</code> from priors, exponentiate to get <code>rp_rs</code>, call <code>transit_model_jax</code>, and use <code>numpyro.sample("obs", ...)</code> with a Gaussian likelihood.</li>
<li>Define <code>gp_transit_model(t, y, yerr)</code>: same transit priors, then compute residuals. Sample GP hyperparameters (<code>log_amp_gp</code>, <code>log_ell_gp</code>), build a Matérn-3/2 kernel, construct a <code>GaussianProcess</code> with <code>diag=yerr**2</code>, and call <code>numpyro.factor("gp", gp.log_probability(residuals))</code>.</li>
<li>Run both fits with NUTS. Use <code>init_to_value</code> for both. Use <code>dense_mass=True</code> for the GP model.</li>
</ol>
</div>
</details>
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 1: JAX differentiability</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
NUTS requires all functions to be differentiable. A hard step function (like <code>jnp.where(z < 1-p, ...)</code>) has zero gradient almost everywhere, which breaks the sampler. The provided <code>transit_model_jax</code> uses <code>jax.nn.sigmoid</code> to smooth the ingress/egress, and <code>jnp.clip(1 - z**2, 1e-12)</code> inside the <code>sqrt</code> to keep gradients finite at the transit limb. You do not need to modify it.
</div>
</details>
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Hint 2: Why <code>numpyro.factor</code> instead of <code>numpyro.sample</code>?</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
In the no-GP model, the likelihood is a product of independent Gaussians, so <code>numpyro.sample("obs", dist.Normal(...), obs=y)</code> works. In the GP model, the likelihood is a single multivariate Gaussian over all data points (the covariance matrix encodes the correlations). <code>tinygp</code> computes this for you via <code>gp.log_probability(residuals)</code>. You inject it into the NumPyro model with <code>numpyro.factor("gp", ...)</code>, which adds an arbitrary log-density term to the joint log-probability.
</div>
</details>
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Partial solution: no-GP model</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
```python
def no_gp_transit_model(t, y, yerr):
    T0        = numpyro.sample("T0", dist.Normal(0.0, 0.05))
    log_rp_rs = numpyro.sample("log_rp_rs", dist.Normal(jnp.log(0.1), 0.3))
    rp_rs     = jnp.exp(log_rp_rs)
    flux_model = transit_model_jax(t, T0, rp_rs)
    numpyro.sample("obs", dist.Normal(flux_model, yerr), obs=y)
```
This assumes the noise is white and Gaussian, which it isn't.

</div>
</details>
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Partial solution: GP model</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
```python
def gp_transit_model(t, y, yerr):
    T0        = numpyro.sample("T0", dist.Normal(0.0, 0.05))
    log_rp_rs = numpyro.sample("log_rp_rs", dist.Normal(jnp.log(0.1), 0.3))
    rp_rs     = jnp.exp(log_rp_rs)

    residuals = y - transit_model_jax(t, T0, rp_rs)

    log_amp_gp = numpyro.sample("log_amp_gp", dist.Normal(jnp.log(0.003), 1.0))
    log_ell_gp = numpyro.sample("log_ell_gp", dist.Normal(jnp.log(0.08), 0.5))
    amp_gp = jnp.exp(log_amp_gp)
    ell_gp = jnp.exp(log_ell_gp)

    kernel = amp_gp**2 * kernels.Matern32(scale=ell_gp)
    gp = GaussianProcess(kernel, t, diag=yerr**2)
    numpyro.factor("gp", gp.log_probability(residuals))
```
Mathematically identical to passing <code>mean=flux_model</code> to <code>GaussianProcess</code>, but avoids shape issues during NumPyro's tracing.
</div>
</details>
<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Full solution (models + MCMC)</summary>
<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">
    
```python
# --- Fit 1: No GP ---
def no_gp_transit_model(t, y, yerr):
    T0        = numpyro.sample("T0", dist.Normal(0.0, 0.05))
    log_rp_rs = numpyro.sample("log_rp_rs", dist.Normal(jnp.log(0.1), 0.3))
    rp_rs     = jnp.exp(log_rp_rs)
    flux_model = transit_model_jax(t, T0, rp_rs)
    numpyro.sample("obs", dist.Normal(flux_model, yerr), obs=y)

# --- Fit 2: With GP ---
def gp_transit_model(t, y, yerr):
    T0        = numpyro.sample("T0", dist.Normal(0.0, 0.05))
    log_rp_rs = numpyro.sample("log_rp_rs", dist.Normal(jnp.log(0.1), 0.3))
    rp_rs     = jnp.exp(log_rp_rs)

    residuals = y - transit_model_jax(t, T0, rp_rs)

    log_amp_gp = numpyro.sample("log_amp_gp", dist.Normal(jnp.log(0.003), 1.0))
    log_ell_gp = numpyro.sample("log_ell_gp", dist.Normal(jnp.log(0.08), 0.5))
    amp_gp = jnp.exp(log_amp_gp)
    ell_gp = jnp.exp(log_ell_gp)

    kernel = amp_gp**2 * kernels.Matern32(scale=ell_gp)
    gp = GaussianProcess(kernel, t, diag=yerr**2)
    numpyro.factor("gp", gp.log_probability(residuals))

# --- Run both ---
init_vals = {"T0": jnp.array(0.0), "log_rp_rs": jnp.array(jnp.log(0.1))}

mcmc_no_gp = MCMC(
    NUTS(no_gp_transit_model, init_strategy=init_to_value(values=init_vals)),
    num_warmup=200, num_samples=1000, num_chains=1
)
mcmc_no_gp.run(random.PRNGKey(5), t_tr_jax, f_obs_tr_jax, sigma_tr_jax)

mcmc_gp = MCMC(
    NUTS(gp_transit_model, dense_mass=True, init_strategy=init_to_value(values=init_vals)),
    num_warmup=200, num_samples=1000, num_chains=1
)
mcmc_gp.run(random.PRNGKey(6), t_tr_jax, f_obs_tr_jax, sigma_tr_jax)
```
</div>
</details>
</div>

In [ ]:
# ===========================================================
# Posterior predictive comparison: transit fits overlaid on data
# ===========================================================
samples_no_gp = mcmc_no_gp.get_samples()
samples_gp    = mcmc_gp.get_samples()

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(t_tr, f_obs_tr, yerr=sigma_tr, fmt='.', color='k',
            ecolor='grey', capsize=0, ms=3, alpha=0.5, zorder=1)

# No-GP: posterior predictive is just the transit model at each sample
for j in range(50):
    rp_j = jnp.exp(samples_no_gp["log_rp_rs"][j])
    f_draw = transit_model_jax(t_tr_jax, samples_no_gp["T0"][j], rp_j)
    ax.plot(t_tr, np.array(f_draw), color='tab:orange', lw=0.3, alpha=0.15)

# With GP: posterior predictive = transit + GP-predicted systematics
# For each posterior sample, condition the GP on the residuals and
# add the GP mean prediction back to the transit model.
t_fine = jnp.linspace(-0.2, 0.2, 300)
for j in range(50):
    T0_j  = samples_gp["T0"][j]
    rp_j  = jnp.exp(samples_gp["log_rp_rs"][j])
    amp_j = jnp.exp(samples_gp["log_amp_gp"][j])
    ell_j = jnp.exp(samples_gp["log_ell_gp"][j])
    
    flux_j = transit_model_jax(t_tr_jax, T0_j, rp_j)
    resid  = f_obs_tr_jax - flux_j
    k_j    = amp_j**2 * kernels.Matern32(scale=ell_j)
    gp_j   = GaussianProcess(k_j, t_tr_jax, diag=sigma_tr_jax**2)
    _, cond_j = gp_j.condition(resid, t_fine)
    
    prediction = transit_model_jax(t_fine, T0_j, rp_j) + cond_j.loc
    ax.plot(t_fine, np.array(prediction), color='tab:blue', lw=0.3, alpha=0.15)

# Ground truth
t_truth_fine = jnp.linspace(-0.2, 0.2, 500)
ax.plot(t_fine_plot, flux_transit_fine + systematic_fine, 'black', lw=1.5, alpha=0.7, 
             label='Transit + systematics (truth)', ls = '--')

ax.plot([], [], color='tab:orange', lw=2, label='Without GP')
ax.plot([], [], color='tab:blue', lw=2, label='With GP')
ax.legend(fontsize=9)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Relative flux')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================
# Corner plot: compare posteriors on T0 and Rp/Rs
# ===========================================================
rp_no_gp = np.exp(np.array(samples_no_gp["log_rp_rs"]))
rp_gp    = np.exp(np.array(samples_gp["log_rp_rs"]))

corner_data_no_gp = np.column_stack([np.array(samples_no_gp["T0"]), rp_no_gp])
corner_data_gp    = np.column_stack([np.array(samples_gp["T0"]),    rp_gp])

fig_corner = corner.corner(
    corner_data_no_gp, labels=[r"$T_0$ [days]", r"$R_p/R_\star$"],
    color='tab:orange', truths=[T0_true, rp_rs_true],
    plot_datapoints=True, plot_density=False,
    levels=(0.68, 0.95), hist_kwargs={'density': True}
)
corner.corner(
    corner_data_gp, fig=fig_corner,
    color='tab:blue', plot_datapoints=True, plot_density=False,
    levels=(0.68, 0.95), hist_kwargs={'density': True}
)
plt.suptitle('Orange = no GP (biased, overconfident)\n'
             'Blue = with GP (honest)', fontsize=11, y=1.02)
plt.show()

# Print summary
T0_no = np.array(samples_no_gp["T0"])
T0_gp = np.array(samples_gp["T0"])
print(f"T0:    no GP = {np.median(T0_no):.4f} +/- {np.std(T0_no):.4f}")
print(f"       GP    = {np.median(T0_gp):.4f} +/- {np.std(T0_gp):.4f}")
print(f"       true  = {T0_true:.4f}")
print()
print(f"Rp/Rs: no GP = {np.median(rp_no_gp):.4f} +/- {np.std(rp_no_gp):.4f}")
print(f"       GP    = {np.median(rp_gp):.4f} +/- {np.std(rp_gp):.4f}")
print(f"       true  = {rp_rs_true:.4f}")

In [ ]:
# ===========================================================
# Residual comparison: with and without GP
# ===========================================================

# Median parameter values
T0_med_no    = jnp.median(samples_no_gp["T0"])
rp_med_no    = jnp.exp(jnp.median(samples_no_gp["log_rp_rs"]))
T0_med_gp    = jnp.median(samples_gp["T0"])
rp_med_gp    = jnp.exp(jnp.median(samples_gp["log_rp_rs"]))
amp_med_gp   = jnp.exp(jnp.median(samples_gp["log_amp_gp"]))
ell_med_gp   = jnp.exp(jnp.median(samples_gp["log_ell_gp"]))

# No-GP residuals: data minus best-fit transit
resid_no_gp = f_obs_tr - np.array(transit_model_jax(t_tr_jax, T0_med_no, rp_med_no))

# GP residuals: data minus best-fit transit minus GP predictive mean
resid_gp_transit = f_obs_tr - np.array(transit_model_jax(t_tr_jax, T0_med_gp, rp_med_gp))
k_med = amp_med_gp**2 * kernels.Matern32(scale=ell_med_gp)
gp_med = GaussianProcess(k_med, t_tr_jax, diag=sigma_tr_jax**2)
_, cond_med = gp_med.condition(jnp.array(resid_gp_transit), t_tr_jax)
resid_gp_full = np.array(jnp.array(resid_gp_transit) - cond_med.loc)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, sharey=True)

axes[0].errorbar(t_tr, resid_no_gp, yerr=sigma_tr, fmt='.', color='tab:orange',
                 ecolor='grey', capsize=0, ms=3, alpha=0.7)
axes[0].axhline(0, color='grey', ls='--')
axes[0].set_ylabel('Residual')
axes[0].set_title('No GP: residuals show clear correlated structure')

axes[1].errorbar(t_tr, resid_gp_full, yerr=sigma_tr, fmt='.', color='tab:blue',
                 ecolor='grey', capsize=0, ms=3, alpha=0.7)
axes[1].axhline(0, color='grey', ls='--')
axes[1].set_ylabel('Residual')
axes[1].set_xlabel('Time (days)')
axes[1].set_title('With GP: residuals consistent with white noise')

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<b>Question:</b> Look at the corner plot and the residuals. How do the two fits differ?

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

The no-GP fit (orange) gives tight posteriors that miss the true values. The systematics bias $T_0$ and $R_p/R_\star$, and because the model thinks the noise is white, it dramatically underestimates the uncertainties. The residuals have correlated structure remains that the model failed to account for.

The GP fit (blue) gives wider posteriors that contain the true values. It is less precise but more accurate. The GP absorbs the systematics, and the resulting uncertainties correctly reflect how much the data actually constrain the transit parameters.

The lesson is that ignoring correlated noise does not just add scatter to your answer but it gives you the wrong answer with high confidence.

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>§3.4 Exercise: Summary</h3>

The full generative model says the data are drawn from:

$$\mathbf{y} \sim \mathcal{N}\bigl(\mathbf{m}(\boldsymbol{\theta}),\; C_\phi\bigr)$$

where $\mathbf{m}(\boldsymbol{\theta})$ is the transit model (parameterised by $T_0$ and $R_p/R_\star$) and $C_{ij} = k_\phi(t_i, t_j) + \sigma_i^2\delta_{ij}$. The kernel $k_\phi$ (here Matérn-3/2) encodes the correlated systematics; the $\sigma_i^2$ on the diagonal is the measurement uncertainty. The log-likelihood is:

$$\ln p(\mathbf{y} \mid \boldsymbol{\theta}, \boldsymbol{\phi}) = -\tfrac{1}{2}(\mathbf{y} - \mathbf{m})^T C^{-1}(\mathbf{y} - \mathbf{m}) - \tfrac{1}{2}\ln\det C - \tfrac{N}{2}\ln 2\pi$$

In the code, this maps directly onto:

<ul>
<li><code>residuals = y - transit_model_jax(t, T0, rp_rs)</code> $\;\longrightarrow\; \mathbf{y} - \mathbf{m}(\boldsymbol{\theta})$</li>
<li><code>kernel = amp_gp**2 * kernels.Matern32(scale=ell_gp)</code> $\;\longrightarrow\; k_\phi(t_i, t_j)$</li>
<li><code>GaussianProcess(kernel, t, diag=yerr**2)</code> $\;\longrightarrow\; C_\phi = K_\phi + \sigma^2 I$</li>
<li><code>gp.log_probability(residuals)</code> $\;\longrightarrow\;$ the full log-likelihood above</li>
</ul>

All four parameters ($T_0$, $R_p/R_\star$, amplitude, length scale) are free inside the model function and explored jointly by NUTS.

The residual plots above are a post-inference diagnostic. For the no-GP fit, the residuals $r_i = y_i - m(t_i, \hat{\boldsymbol{\theta}})$ show clear correlated structure: the systematics the model failed to account for. For the GP fit, we additionally subtract the GP predictive mean (via <code>gp.condition</code>) to get $r_i = (y_i - m(t_i, \hat{\boldsymbol{\theta}})) - \mu_\mathrm{GP}(t_i)$, which should be consistent with white noise if the GP has captured the systematics.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<hr>

<h1>§3.5 Application: Spectral Continuum Normalisation </h1>

GPs are not limited to time-series. The input domain can be anything, such as wavelength, spatial coordinates, or any combination. Here, the domain is wavelength.

Continuum normalisation is a standard step in stellar spectroscopy. You need to divide the observed spectrum by the smooth continuum to isolate the absorption lines. The usual approach is a polynomial fit, but the polynomial order is somewhat arbitrary and this process (by default) doesn't provide uncertainty on the continuum, and it can be biased by deep absorption lines.

A GP provides a principled alternative. The idea:
<ol>
<li>Mask pixels that are clearly in absorption lines (e.g., sigma-clipping).</li>
<li>Fit a GP to the unmasked (continuum) pixels. Use a long-lengthscale kernel (the continuum varies on scales of ~100 Å, not ~1 Å).</li>
<li>Predict the continuum everywhere, including inside the masked lines.</li>
<li>Divide: normalised spectrum = observed / GP continuum.</li>
</ol>

The GP gives you an uncertainty estimate on the continuum level, which propagates into the uncertainties on your line measurements.

Below, we have generated a synthetic spectra with a number of absorption lines. We have put a 2nd order polynomial on top to get the unnormalised spectra. We did this because it is easy to do, but in principle, the distortion could be anything. In any case, the GP doesn't care about the functional form of the distoriton

</div>

In [ ]:
# ── Generate a synthetic stellar spectrum ──
np.random.seed(42)

# Wavelength grid (Angstroms)
lam = np.linspace(4800, 5200, 800)

# Smooth continuum: gentle slope + broad curvature
continuum_true = 1.0 + 0.1 * (lam - 5000) / 200 - 0.03 * ((lam - 5000) / 200)**2

# Absorption lines (Gaussians)
def absorption_line(lam, lam_0, depth, width):
    return depth * np.exp(-0.5 * ((lam - lam_0) / width)**2)

lines = [
    (4861, 0.6, 3.0),   # H-beta
    (4922, 0.15, 1.5),   # Fe I
    (5007, 0.25, 2.0),   # [O III] (pretend it's absorption for demo)
    (5051, 0.10, 1.0),   # Fe I
    (5110, 0.18, 1.8),   # Fe I
    (5167, 0.35, 2.5),   # Mg I triplet
    (5173, 0.30, 2.5),   # Mg I triplet
    (5184, 0.28, 2.5),   # Mg I triplet
]

absorption = np.zeros_like(lam)
for l0, d, w in lines:
    absorption += absorption_line(lam, l0, d, w)

flux_true = continuum_true * (1 - absorption)

# Add noise
snr = 50
sigma_spec = continuum_true / snr
flux_obs = flux_true + sigma_spec * np.random.randn(len(lam))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lam, flux_obs, 'k-', lw=0.5, alpha=0.7, label='Observed')
ax.plot(lam, continuum_true, 'tab:orange', lw=2, ls='--', label='True continuum')
ax.set_xlabel('Wavelength (Å)')
ax.set_ylabel('Flux')
ax.legend()
ax.set_title('Synthetic stellar spectrum')
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<b>Task:</b>
<ol>
<li>Identify continuum pixels using sigma-clipping (mask anything more than 2σ below a running median).</li>
<li>Fit a GP with a long-lengthscale Matérn-5/2 kernel (if you think this is valid! Draw some prior samples to see if it passes the vibe check) to the unmasked pixels only.</li>
<li>Predict the continuum at all wavelengths.</li>
<li>Divide to get the normalised spectrum.</li>
</ol>

Use <code>scipy.optimize.minimize</code> for the hyperparameters here. This is a simpler problem (only two hyperparameters, a smooth marginal likelihood surface) where optimisation is adequate. We use NumPyro for the more complex models (§3.4) where the posterior geometry is harder and uncertainty propagation matters.

</div>

In [ ]:
###############################################################
# Step 1: Identify continuum pixels                           #
# TODO: Use a running median filter to get a rough continuum, #
#       then mask pixels that are >2*sigma below it           #
#      (use median_filter we imported from scipy.ndimage)     #
###############################################################

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
# Step 1: Sigma-clip to find continuum
rough_continuum = median_filter(flux_obs, size=50)
residuals = flux_obs - rough_continuum
continuum_mask = residuals > -2 * sigma_spec.mean()
```

</div>
</details>

</div>

In [ ]:
#################################################################
# Step 2: Build and optimise GP on continuum pixels             #
# TODO:                                                         #
#   - Use kernels.Matern52(scale=ell) with amplitude            #
#   - Optimise log_amp and log_ell                              #
#   - Fit ONLY to lam[continuum_mask], flux_obs[continuum_mask] #
#################################################################

def neg_log_ml(params, x, y, yerr):
    """Negative log marginal likelihood for GP hyperparameter optimisation."""
    # extract params
    # define kernel
    # create gp
    return # prob

# TODO: Optimise


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
# Step 2: Optimise GP
lam_cont = jnp.array(lam[continuum_mask])
flux_cont = jnp.array(flux_obs[continuum_mask])
yerr_cont = jnp.array(sigma_spec[continuum_mask])

result = scipy_minimize(
    neg_log_ml, x0=[np.log(0.1), np.log(50.0)],
    args=(lam_cont, flux_cont, yerr_cont),
    method='L-BFGS-B'
)
amp_opt, ell_opt = np.exp(result.x)
print(f"Optimised: amp = {amp_opt:.4f}, ℓ = {ell_opt:.1f} Å")
```

</div>
</details>

</div>

In [ ]:
#############################################
# Step 3: Predict continuum everywhere      #
# TODO:                                     #
#   - Build GP at optimised hyperparameters #
#   - Condition on continuum pixels         #
#   - Predict at ALL wavelengths            #
#############################################

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
# Step 3: Predict everywhere
k_opt = amp_opt**2 * kernels.Matern52(scale=ell_opt)
gp_cont = GaussianProcess(k_opt, lam_cont, diag=yerr_cont**2)
_, cond_cont = gp_cont.condition(flux_cont, jnp.array(lam))
cont_mean = cond_cont.loc
cont_std = jnp.sqrt(cond_cont.variance)
```

</div>
</details>

</div>

In [ ]:
#####################################################
# Step 4: Plot and normalise                        #
# TODO:                                             #
#   - Plot observed spectrum + GP continuum ± 1σ    #
#   - Plot normalised spectrum (flux_obs / GP_mean) #
#####################################################

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Solution</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python
# Step 4: Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top: spectrum + continuum
axes[0].plot(lam, flux_obs, 'k-', lw=0.5, alpha=0.7, label='Observed')
axes[0].plot(lam, np.array(cont_mean), 'tab:blue', lw=2, label='GP continuum')
axes[0].fill_between(lam, np.array(cont_mean - cont_std), np.array(cont_mean + cont_std),
                     alpha=0.2, color='tab:blue')
axes[0].plot(lam, continuum_true, 'tab:orange', ls='--', lw=1.5, label='True continuum')
axes[0].scatter(lam[~continuum_mask], flux_obs[~continuum_mask], 
                c='tab:red', s=2, alpha=0.3, label='Masked (lines)')
axes[0].legend(fontsize=8)
axes[0].set_ylabel('Flux')

# Bottom: normalised spectrum
normalised = flux_obs / np.array(cont_mean)
axes[1].plot(lam, normalised, 'k-', lw=0.5)
axes[1].axhline(1, color='grey', ls='--', alpha=0.5)
axes[1].set_xlabel('Wavelength (Å)')
axes[1].set_ylabel('Normalised flux')
axes[1].set_ylim([0.3, 1.15])

plt.tight_layout()
plt.show()
```

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

You should see the GP continuum dipping into the absorption lines, particularly the deeper ones like H$\beta$ and the Mg I triplet. The continuum estimate is being pulled below the true continuum in the gaps.

<b>Why this happens:</b> The sigma-clipping mask removes the cores of the absorption lines, but the <i>wings</i> survive. These wing points sit slightly below the true continuum and pass the <code>residuals > -2 * sigma</code> threshold. The GP treats them as legitimate continuum points, and since the optimised length scale (~70 Å) is short enough to follow structure on the scale of the line spacing, the GP interpolates through the depressed wing points and dips into the gaps.

<b>The fix:</b> Dilate the mask. After sigma-clipping, expand each rejected region by a few pixels on either side to remove the contaminated wings:

```python
    from scipy.ndimage import binary_dilation
    continuum_mask = ~binary_dilation(~continuum_mask, iterations=5)
```


(This function should already be imported)

With the wings removed, the GP trains only on genuinely line-free regions. The optimiser then naturally finds a longer length scale (the data no longer reward short-scale flexibility), and the continuum estimate stays flat across the line gaps.


<b>Aside:</b> MAP optimisation found a single point in parameter space and committed to it. If you had run MCMC or nested sampling instead, the posterior on the length scale would likely have been broad and multimodal, hinting that the short length scale is not the only (or best) explanation. Point estimates can confidently land in misleading regions of parameter space, especially when the data are contaminated. Full posterior exploration is more robust to this, because it reveals the uncertainty in the hyperparameters rather than hiding it.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<b>Question:</b> What happens if you don't mask the absorption lines before fitting the GP?

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

The GP continuum estimate will be pulled <i>down</i> near deep absorption lines. The normalised line depths will be underestimated. Try it: re-run the fit without the mask and see how the normalised Mg triplet changes.

</div>
</details>

<b>Question:</b> What happens if the GP length scale is too short (e.g., $\ell \sim 5$ Å)?

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

The GP starts to fit structure on scales comparable to the line widths. The "continuum" becomes wiggly and traces the wings of the absorption lines. Again, the normalisation is biased. A physically motivated lower bound on $\ell$ — "the continuum shouldn't vary on scales shorter than ~50 Å" — is the spectral equivalent of putting an informed prior on the GP evolution timescale in transit fitting.

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§3.5 Demonstration: Gaussian Processes in Two Dimensions</h2>

Everything so far has been one-dimensional: flux as a function of time. But nothing in the GP framework requires scalar inputs. The kernel $k(\mathbf{x}_i, \mathbf{x}_j)$ only needs a notion of distance between inputs. Typically, this is the Euclidean norm $\|\mathbf{x}_i - \mathbf{x}_j\|_2$, and the covariance matrix is built exactly as before.

We demonstrate this with a simulated galaxy velocity field: the line-of-sight velocity of ionised gas as a function of sky position $(x, y)$, as measured by an Integral Field Unit (IFU) spectrograph. IFU data products are a natural fit for 2D GP regression: coverage is spatially irregular (fixed fibre layout), noise increases with radius as surface brightness falls off, and the underlying field is smooth.

Our goals for this section:
<ol>
<li>Build intuition for how the GP generalises from 1D to 2D with minimal code change.</li>
<li>Reconstruct a continuous velocity field from sparse, noisy measurements.</li>
<li>Visualise the posterior uncertainty and identify where the GP deviates from truth.</li>
</ol>

I will say now, IFUs and galaxy velocity are things that I know little about. There is a decent chance that none of this makes astrophysical sense. Hopefully it still makes mathematical sense.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 1: IFU Fibre Layout</h3>

We will generate a hexagonal fibre layout analytically. For a bundle with $n_\text{rings}$ rings and inter-fibre pitch $p$, the $r$-th ring has $6r$ fibres placed by walking around a hexagon.

</div>

In [ ]:
def hex_ifu(n_rings, pitch):
    """
    Generate fibre positions for a hexagonal IFU bundle.

    Parameters
    ----------
    n_rings : int
        Number of concentric rings around the central fibre.
    pitch : float
        Centre-to-centre fibre spacing (kpc in our simulation).

    Returns
    -------
    positions : ndarray of shape (N, 2)
    """
    positions = [(0.0, 0.0)]  # central fibre

    for ring in range(1, n_rings + 1):
        for side in range(6):
            for step in range(ring):
                # Start at a corner of this ring's hexagon, then step along each side.
                angle_corner = np.radians(60 * side + 30)
                angle_side   = np.radians(60 * (side + 2) + 30)
                x = ring * pitch * np.cos(angle_corner) + step * pitch * np.cos(angle_side)
                y = ring * pitch * np.sin(angle_corner) + step * pitch * np.sin(angle_side)
                positions.append((x, y))

    return np.array(positions)


ifu_pos = hex_ifu(n_rings=6, pitch=2.0)
x_obs, y_obs = ifu_pos[:, 0], ifu_pos[:, 1]
print(f"Number of fibres: {len(x_obs)}")

plt.figure(figsize=(5, 5))
plt.scatter(x_obs, y_obs, s=30)
plt.gca().set_aspect('equal')
plt.title("IFU fibre positions")
plt.xlabel("x (kpc)")
plt.ylabel("y (kpc)")
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 2: Constructing a Synthetic Velocity Field</h3>

We build a "ground truth" velocity field with three components:

<ol>
<li><b>Rotation curve.</b> $v_\text{rot}(r) = v_\text{max} \, r / (r + r_\text{turn})$. This rises linearly near the centre and asymptotes to $v_\text{max}$, approximating observed disk galaxy rotation curves.</li>
<li><b>Radial taper.</b> Surface brightness falls off with radius. We multiply by $e^{-r/r_\text{decay}}$ to suppress the signal in the outskirts.</li>
<li><b>Kinematic anomaly.</b> A localised Gaussian bump mimics infalling gas or a bar-driven oval distortion. In real data, identifying these anomalies is a primary science goal.</li>
</ol>

In this demonstration we have access to the truth. In real observations, you do not.

</div>

In [ ]:
def radial_taper(r, r_decay=8.0):
    """Exponential envelope suppressing signal at large radii (surface-brightness falloff)."""
    return np.exp(-r / r_decay)


def true_velocity_field(x, y):
    """
    Synthetic IFU line-of-sight velocity field (km/s).

    Components: arctan rotation curve + line-of-sight projection
                + radial taper + localised Gaussian anomaly.
    """
    v_max  = 200.0           # asymptotic rotation speed  (km/s)
    r_turn = 3.0             # turnover radius             (kpc)
    inc    = np.radians(60)  # disk inclination from face-on

    r     = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)

    v_rot  = v_max * r / (r + r_turn)           # rotation curve
    v_los  = v_rot * np.sin(theta) * np.sin(inc)  # project onto line of sight
    v_los *= radial_taper(r)                    # apply surface-brightness taper

    # Kinematic anomaly at (4, 3) kpc, width 2.5 kpc
    d2     = (x - 4.0)**2 + (y - 3.0)**2
    v_los += -80.0 * np.exp(-0.5 * d2 / 2.5**2)

    return v_los


# Evaluate on a fine grid for visualisation
N_grid = 80
x_fine = np.linspace(-15, 15, N_grid)
y_fine = np.linspace(-15, 15, N_grid)
X_fine, Y_fine = np.meshgrid(x_fine, y_fine)
V_true = true_velocity_field(X_fine, Y_fine)

plt.figure(figsize=(6, 5))
plt.pcolormesh(X_fine, Y_fine, V_true, cmap='RdBu_r', shading='auto')
plt.colorbar(label="km/s")
plt.gca().set_aspect('equal')
plt.title("True velocity field (hidden in real observations)")
plt.xlabel("x (kpc)")
plt.ylabel("y (kpc)")
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 3: Heteroscedastic Noise Model</h3>

We model this as:

$$\sigma(r) = \sigma_0 \, e^{r / r_\text{scale}}$$

This makes the noise heteroscedastic: each observation $i$ has its own known noise level $\sigma_i$. In the GP likelihood, this enters as a diagonal but non-uniform noise covariance $\mathbf{\Sigma}_n = \text{diag}(\sigma_1^2, \ldots, \sigma_N^2)$. In <code>tinygp</code>, we pass this as <code>diag=sig_norm**2</code> — a per-point array rather than a scalar. Heteroscedastic noise requires no additional complexity in the GP framework.

</div>

In [ ]:
sigma_base = 1.8   # central velocity uncertainty (km/s)
r_scale    = 6.0   # radial scale of noise growth  (kpc)

r_obs     = np.sqrt(x_obs**2 + y_obs**2)
sigma_obs = sigma_base * np.exp(r_obs / r_scale)   # per-fibre noise

# Draw noisy observations
np.random.seed(42)
v_true_obs = true_velocity_field(x_obs, y_obs)
v_obs      = v_true_obs + sigma_obs * np.random.randn(len(x_obs))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc = axes[0].scatter(x_obs, y_obs, c=v_obs, cmap='RdBu_r', s=500,
                     edgecolors='k', linewidths=0.3, marker='H')
plt.colorbar(sc, ax=axes[0], label="km/s")
axes[0].set_aspect('equal')
axes[0].set_title("Observed IFU velocities")
axes[0].set_xlabel("x (kpc)");  axes[0].set_ylabel("y (kpc)")

R_fine  = np.sqrt(X_fine**2 + Y_fine**2)
SNR_map = np.abs(V_true) / (sigma_base * np.exp(R_fine / r_scale))
im = axes[1].pcolormesh(X_fine, Y_fine, np.clip(SNR_map, 0, 20), cmap='magma', shading='auto')
plt.colorbar(im, ax=axes[1], label="SNR")
axes[1].set_aspect('equal')
axes[1].set_title("Signal-to-noise ratio (truth / noise)")
axes[1].set_xlabel("x (kpc)")

plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 4: The 2D GP Model</h3>

We place a zero-mean GP prior on the velocity field:

$$v(\mathbf{x}) \sim \mathcal{GP}\!\left(0,\; k(\mathbf{x}, \mathbf{x}')\right), \qquad \mathbf{x} = (x, y)^\top \in \mathbb{R}^2$$

with a squared-exponential kernel:

$$k(\mathbf{x}, \mathbf{x}') = \alpha^2 \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

<b>The only code change from 1D is the shape of the input array.</b> Instead of a length-$N$ vector, we pass an $(N, 2)$ matrix. <code>tinygp</code> automatically computes $\|\mathbf{x}_i - \mathbf{x}_j\|_2$ and evaluates the kernel on that distance. Everything else is identical to the 1D case.

<b>Numerical normalisation.</b> With velocity amplitudes of order 200 km/s and noise at the few km/s level, the covariance matrix spans large dynamic range. We normalise the observations to zero mean and unit variance before passing them to the GP, then undo the transformation on the predictions. This is numerically prudent and does not change the model.

</div>

In [ ]:
coords = jnp.array(np.column_stack([x_obs, y_obs]))  # shape (N, 2)

# Normalise for numerical stability
v_mean   = float(np.mean(v_obs))
v_std    = float(np.std(v_obs))
v_norm   = jnp.array((v_obs - v_mean) / v_std)
sig_norm = jnp.array(sigma_obs / v_std)


@jax.jit
@jax.value_and_grad
def neg_log_likelihood(log_params):
    """
    Negative log marginal likelihood.

    Parameters are log-transformed to enforce positivity and keep
    the optimiser on a roughly uniform scale.

    log_params = [log(amplitude), log(length scale)]
    """
    log_amp, log_ell = log_params
    kernel = jnp.exp(log_amp)**2 * kernels.ExpSquared(scale=jnp.exp(log_ell))
    # diag=sig_norm**2 sets per-point (heteroscedastic) noise variance
    gp     = GaussianProcess(kernel, coords, diag=sig_norm**2)
    return -gp.log_probability(v_norm)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Exercise — Optimise the Hyperparameters</h3>

Use <code>scipy_minimize</code> with analytic gradients (from JAX via <code>value_and_grad</code>) to find the maximum marginal likelihood hyperparameters $\hat{\alpha}$ and $\hat{\ell}$.

Fill in the skeleton below. Then interpret the recovered length scale $\hat{\ell}$ physically: is it consistent with the spatial scales visible in the velocity field?

</div>

In [ ]:
# ── SKELETON ────────────────────────────────────────────────────────────────
# scipy_minimize expects a Python callable returning (float, np.array).
# We wrap neg_log_likelihood which returns JAX arrays.

def objective(params):
    val, grad = neg_log_likelihood(jnp.array(params))
    return float(val), np.array(grad)

# TODO: choose a sensible initial guess for [log(amp), log(ell)].
# After normalisation the velocity amplitude is ~1; the spatial coherence
# is on scales of a few kpc.
x0 = # ???

# TODO: call scipy_minimize with jac=True and method="L-BFGS-B"
res = # ???

# TODO: recover amp and ell (remember: parameters are log-transformed)
amp, ell = # ???
print(f"Optimised amplitude:    {amp:.3f} (normalised units)")
print(f"Optimised length scale: {ell:.2f} kpc")

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<b>Solution</b>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python

def objective(params):
    val, grad = neg_log_likelihood(jnp.array(params))
    return float(val), np.array(grad)

x0 = [0.0, np.log(5.0)]   # amp ~ 1 (normalised), ell ~ 5 kpc

res = scipy_minimize(objective, x0=x0, jac=True, method="L-BFGS-B")

amp, ell = np.exp(res.x)
print(f"Optimised amplitude:    {amp:.3f}")
print(f"Optimised length scale: {ell:.2f} kpc")
```

The recovered length scale is typically 5–9 kpc — consistent with the rotation curve turnover radius (3 kpc) and the anomaly width (2.5 kpc). The GP discovers the dominant smoothness scale directly from the data via the marginal likelihood.

</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 5: Conditioning and Prediction</h3>

With hyperparameters fixed, we condition the GP on the observations to get the posterior predictive distribution at every fine-grid point. 
</div>

In [ ]:
# Rebuild GP with optimised hyperparameters
kernel_opt = ...
gp_opt     = ...

# Flatten prediction grid: (80, 80) -> (6400, 2)
coords_fine = jnp.array(np.column_stack([X_fine.ravel(), Y_fine.ravel()]))

# Condition on observations; returns (log_prob, conditional_gp_object)
...

# Un-normalise predictions
V_pred    = ...
V_std_map = ...

print(f"Predicted velocity range: [{V_pred.min():.1f}, {V_pred.max():.1f}] km/s")
print(f"Posterior std range:       [{V_std_map.min():.1f}, {V_std_map.max():.1f}] km/s")

<div style="max-width: 800px; margin: 0 auto; text-align: justify; padding: 20px; background-color: rgba(0, 64, 133, 0.08); border: 2px solid rgba(0, 64, 133, 0.4); border-radius: 10px;">

<b>Solution</b>

<details style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold; padding: 10px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">Click to reveal</summary>

<div style="margin-top: 10px; padding: 15px; background-color: rgba(0, 64, 133, 0.12); border: 1px solid rgba(0, 64, 133, 0.4); border-radius: 5px;">

```python

# Rebuild GP with optimised hyperparameters
kernel_opt = amp**2 * kernels.ExpSquared(scale=ell)
gp_opt     = GaussianProcess(kernel_opt, coords, diag=sig_norm**2)

# Flatten prediction grid: (80, 80) -> (6400, 2)
coords_fine = jnp.array(np.column_stack([X_fine.ravel(), Y_fine.ravel()]))

# Condition on observations; returns (log_prob, conditional_gp_object)
_, cond = gp_opt.condition(v_norm, coords_fine)

# Un-normalise predictions
V_pred    = (np.array(cond.loc)         * v_std + v_mean).reshape(N_grid, N_grid)
V_std_map = (np.sqrt(cond.variance)     * v_std         ).reshape(N_grid, N_grid)

print(f"Predicted velocity range: [{V_pred.min():.1f}, {V_pred.max():.1f}] km/s")
print(f"Posterior std range:       [{V_std_map.min():.1f}, {V_std_map.max():.1f}] km/s")
```


</div>
</details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>Step 6: Visualising Results</h3>

The $2 \times 3$ panel below compares (top row) the truth, the sparse observations, and the GP posterior mean; and (bottom row) the posterior standard deviation, the normalised residuals, and the raw residuals.

Again, I am not sure if these plots are relevant astrophysically, but they look cool

</div>

In [ ]:
fig = plt.figure(figsize=(18, 8))
gs  = gridspec.GridSpec(2, 4, width_ratios=[1, 1, 1, 0.05], hspace=0.35, wspace=0.05)

ax1  = fig.add_subplot(gs[0, 0]);  ax2  = fig.add_subplot(gs[0, 1])
ax3  = fig.add_subplot(gs[0, 2]);  cax1 = fig.add_subplot(gs[0, 3])
ax4  = fig.add_subplot(gs[1, 0]);  ax5  = fig.add_subplot(gs[1, 1])
ax6  = fig.add_subplot(gs[1, 2]);  cax2 = fig.add_subplot(gs[1, 3])

vmax   = 80
norm_v = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

V_res   = V_true - V_pred
res_sig = V_res / V_std_map

# Top row: truth | data | GP mean
ax1.pcolormesh(X_fine, Y_fine, V_true,  cmap='RdBu_r', norm=norm_v, shading='auto')
ax1.set_title("Truth")

ax2.scatter(x_obs, y_obs, c=v_obs, cmap='RdBu_r', norm=norm_v,
            s=300, edgecolors='k', linewidths=0.3, marker='H')
ax2.set_title("Observed data")

im3 = ax3.pcolormesh(X_fine, Y_fine, V_pred, cmap='RdBu_r', norm=norm_v, shading='auto')
ax3.set_title("GP posterior mean")
fig.colorbar(im3, cax=cax1, label="Velocity (km/s)")

# Bottom row: uncertainty | significance | raw residual
im4 = ax4.pcolormesh(X_fine, Y_fine, V_std_map, cmap='viridis', shading='auto')
ax4.set_title(r"Posterior $\sigma$ (km/s)")
fig.colorbar(im4, cax=cax2, label="km/s")

sig_max = 5
im5 = ax5.pcolormesh(X_fine, Y_fine, res_sig,
                     cmap='RdBu_r', vmin=-sig_max, vmax=sig_max, shading='auto')
ax5.set_title(r"(Truth $-$ GP) / $\sigma_\mathrm{GP}$")
fig.colorbar(im5, ax=ax5, label=r"$\sigma$")

res_max = np.max(np.abs(V_res))
im6 = ax6.pcolormesh(X_fine, Y_fine, V_res,
                     cmap='RdBu_r', vmin=-res_max, vmax=res_max, shading='auto')
ax6.set_title(r"Residual (Truth $-$ GP)  [km/s]")
fig.colorbar(im6, ax=ax6, label="km/s")

for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
    ax.set_aspect('equal')
    ax.set_xlabel('x (kpc)')
    ax.set_ylabel('y (kpc)')

plt.suptitle("2D GP reconstruction of IFU velocity field", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h3>§3.5b A Better Kernel?</h3>

The residual significance map reveals the central failure of the SE kernel: the largest mismatches are concentrated near the origin, where the rotation curve gradient is steepest. The SE kernel assigns a single length scale $\ell$ to the entire field but the velocity field seems to have more than one length scale.

Try and find/create a custom kernel that is a better fit to this data,
</div>